# Chuyên đề 05 · Token hóa âm thanh: dựng lại MOSS-Audio-Tokenizer thu nhỏ, từ sóng âm đến token rồi thành giọng nói

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd05-token-hoa-am-thanh.ipynb)

Notebook đi kèm [Chuyên đề 05 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-hoa-am-thanh/). Mô hình ngôn ngữ chỉ đọc và viết được **token**,
những số nguyên trong một bảng từ vựng (Bài 14). Muốn nó nghe và nói, ta cần một **bộ token hóa âm thanh**: biến sóng
âm thành một chuỗi số nguyên ngắn, rồi dựng lại âm thanh từ chuỗi số ấy. Bài báo
[*MOSS-Audio-Tokenizer*](https://arxiv.org/abs/2602.10934) (MOSI.AI và Đại học Phúc Đán, 2026) đề xuất CAT
(*Causal Audio Tokenizer with Transformer*): chỉ dùng các khối Transformer nhân quả, không có lớp tích chập nào,
huấn luyện mọi thành phần cùng lúc, rồi mở rộng lên 1,6 tỉ tham số và 3 triệu giờ âm thanh. Từ token của nó, các tác
giả dựng một mô hình đọc văn bản thành giọng nói (TTS) hoàn toàn tự hồi quy.

Ta không có 3 triệu giờ âm thanh, nên ta dựng lại **cùng thiết kế ở kích thước chạy được trên CPU**: 3.000 bản ghi
những chữ số tiếng Anh được đọc to, một CAT khoảng một triệu tham số, và một TTS nhỏ "đọc" chữ số. Với mỗi nhận định
của bài báo, ta chạy thí nghiệm và kết luận *khớp*, *không khớp* hay *tùy điều kiện*, như ở Chuyên đề 02 và 03.

**Bạn sẽ làm:**
1. Đếm bit: từ 128.000 bit mỗi giây của sóng âm đến 125 bit mỗi giây của một lớp token.
2. Thấy vì sao so từng mẫu (MSE) là thước đo tồi cho âm thanh, rồi tự viết hàm mất mát mel đa tỉ lệ của bài báo.
3. Tự viết lượng tử hóa vectơ dư (RVQ) và *quantizer dropout*, và thấy vì sao một bộ token hóa phải chịu được việc
   bị cắt bớt lớp.
4. Dựng CAT thu nhỏ, kiểm tra nó chạy được theo luồng, rồi nghe một chữ số đi qua 1, 2, 4 … 32 lớp token.
5. Kiểm chứng các nhận định về huấn luyện và mở rộng, cộng hai điều bài báo không thử.
6. Dựng CAT-TTS thu nhỏ (Temporal + Depth Transformer) và kiểm chứng *Progressive Sequence Dropout*.

**Cần biết trước:** [Bài 14 · Token & embedding](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-embedding/),
[Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/) (khối Transformer nhân
quả), [Bài 18 · ViT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vision-transformer/) (cắt thành mảng),
[Bài 19 · Autoencoder & VAE](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/autoencoder-vae/) và
[Bài 20 · GAN](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/gan/).
**Thời gian chạy:** khoảng 1 phút trên CPU, kể cả tải dữ liệu lần đầu.

Notebook dùng **PyTorch**. Mỗi mục chạy trực tiếp một phiên bản nhỏ, rồi nạp kết quả đầy đủ (3 seed) mà script
`reel/codec/sweep.py` của khóa học đã tính sẵn, cùng mã và cùng seed. Notebook có những ô phát âm thanh: hãy đeo tai
nghe và *nghe*, vì tai người vẫn là thước đo tốt nhất cho một bộ mã hóa âm thanh.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

In [ ]:
import base64
import hashlib
import io
import json
import math
import os
import tarfile
import urllib.request
import wave
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import Audio, display
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=1):
    return vn(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def vn_tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.01 <= v <= 10000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Nhãn trục kiểu Việt Nam; gọi ngay trước plt.show(). Trục log ngắn lấy vạch 1–2–5, trục log dài lấy lũy thừa của 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def play(x, label=""):
    """Phát một đoạn âm thanh 8 kHz ngay trong notebook (giữ nguyên độ to, cắt về [−1, 1])."""
    if label:
        print(label)
    display(Audio(np.clip(np.asarray(x, np.float64), -1, 1), rate=SR, normalize=False))

## 2 · Dữ liệu: 3.000 chữ số được đọc to

[Free Spoken Digit Dataset](https://github.com/Jakobovski/free-spoken-digit-dataset) (FSDD, phiên bản 1.0.10, giấy
phép CC BY-SA 4.0): 6 người, mỗi người đọc mỗi chữ số từ 0 đến 9 năm mươi lần, thu ở 8.000 mẫu mỗi giây (8 kHz), mỗi
mẫu 16 bit. Tệp nén tải từ GitHub; ta kiểm MD5 của *nội dung* 3.000 tệp wav thay vì của tệp nén, vì GitHub có thể
nén lại tệp lưu trữ mà không đổi nội dung. Mỗi bản ghi được chỉnh về cùng một độ to (căn trung bình bình phương, RMS,
bằng 0,1), để độ to không tiết lộ ai đang nói. Ta chia theo số thứ tự của bản ghi như README của bộ dữ liệu: bản 0–4
để kiểm tra, bản 5–9 để kiểm định (chọn siêu tham số), còn lại để huấn luyện. Bốn bản ghi dài hơn 1,28 giây bị bỏ
(ba ở tập huấn luyện, một ở tập kiểm định), nên tập kiểm tra còn đủ 300 bản.

In [ ]:
SR, HOP = 8000, 640  # 8 kHz; một khung token = 640 mẫu = 80 ms, tức 12,5 khung mỗi giây như bài báo
WIN = 16  # khung tối đa của một đoạn: 16 · 80 ms = 1,28 s
SPEAKERS = ("george", "jackson", "lucas", "nicolas", "theo", "yweweler")
FSDD = "https://github.com/Jakobovski/free-spoken-digit-dataset/archive/refs/tags/v1.0.10.tar.gz"
TGZ = Path("fsdd-v1.0.10.tar.gz")
if not TGZ.exists():
    urllib.request.urlretrieve(FSDD, TGZ)


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())

In [ ]:
raw = {}
with tarfile.open(TGZ) as tf:
    for m in tf.getmembers():
        if m.isfile() and m.name.endswith(".wav") and "/recordings/" in m.name:
            raw[m.name.rsplit("/", 1)[1]] = tf.extractfile(m).read()
h = hashlib.md5()
for k in sorted(raw):
    h.update(raw[k])
assert len(raw) == 3000 and h.hexdigest() == "2c93442f2c808e24d4d913f6be24d32c", "FSDD bị hỏng, hãy xóa và tải lại"

waves, digit, spk, idx = [], [], [], []
for name in sorted(raw):
    d, s, i = name[:-4].split("_")
    with wave.open(io.BytesIO(raw[name])) as w:
        assert (w.getnchannels(), w.getsampwidth(), w.getframerate()) == (1, 2, SR)
        x = np.frombuffer(w.readframes(w.getnframes()), "<i2").astype(np.float64) / 32768
    waves.append(np.clip(x * 0.1 / np.sqrt(np.mean(x**2)), -1, 1).astype(np.float32))
    digit.append(int(d))
    spk.append(SPEAKERS.index(s))
    idx.append(int(i))
digit, spk, idx = np.array(digit), np.array(spk), np.array(idx)
n_samples = np.array([len(x) for x in waves])
keep = n_samples <= WIN * HOP
split = np.where(idx < 5, "test", np.where(idx < 10, "val", "train"))
TRAIN, VAL, TEST = (np.where((split == s) & keep)[0] for s in ("train", "val", "test"))
dur = n_samples / SR
print(f"{vn(len(waves), 0)} bản ghi, {vn(dur.sum() / 60, 1)} phút; dài {vn(dur.min(), 2)}–{vn(dur.max(), 2)} s, "
      f"trung vị {vn(np.median(dur), 2)} s")  # fmt: skip
print(f"huấn luyện {vn(len(TRAIN), 0)} · kiểm định {len(VAL)} · kiểm tra {len(TEST)}")
assert (len(TRAIN), len(VAL), len(TEST)) == (2397, 299, 300)

Một bản ghi là một dãy số thực, mỗi số là áp suất không khí tại một thời điểm. Hình dưới vẽ chữ "seven" của một
người trong tập kiểm tra: bên trái là sóng âm (trục ngang là thời gian), bên phải là **phổ đồ** (spectrogram): trục
ngang là thời gian, trục dọc là tần số, màu sáng là năng lượng lớn. Những vạch ngang song song, cách đều nhau ở
phổ đồ là các **họa âm**: bội số của tần số cao độ giọng nói. Khoảng cách giữa chúng cho biết giọng trầm hay cao,
một phần quan trọng của "giọng ai". Các dải sáng rộng hơn (formant) cho biết miệng đang phát âm nguyên âm nào,
tức "nói gì".

In [ ]:
EX = TEST[(digit[TEST] == 7) & (spk[TEST] == SPEAKERS.index("theo"))][0]
x_ex = waves[EX]


def spectrogram(ax, x, title, win=256, vmin=-4):
    S = torch.stft(torch.tensor(x), win, win // 4, window=torch.hann_window(win), return_complex=True)
    S = S.abs().clamp_min(1e-5).log10().numpy()
    ax.imshow(S, origin="lower", aspect="auto", vmin=vmin, vmax=1, cmap="magma", extent=[0, len(x) / SR, 0, SR / 2])
    ax.set(title=title, xlabel="thời gian (s)", ylabel="tần số (Hz)")


fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2))
t = np.arange(len(x_ex)) / SR
a1.plot(t, x_ex, color=BLUE, lw=0.6)
a1.set(title='sóng âm: "seven"', xlabel="thời gian (s)", ylabel="biên độ")
spectrogram(a2, x_ex, "phổ đồ (log₁₀ độ lớn)")
vn_axes(fig)
plt.tight_layout()
plt.show()
play(x_ex, f'chữ số {digit[EX]}, người đọc "{SPEAKERS[spk[EX]]}", bản ghi số {idx[EX]}')

## 3 · Bài báo nói gì, và ta kiểm chứng gì

CAT có ba phần, nối tiếp nhau (Hình 2 của bài báo):

- **Bộ mã hóa** (encoder) cắt sóng âm thành từng mảng 240 mẫu, cho qua các khối Transformer *nhân quả* (mỗi vị trí
  chỉ nhìn về quá khứ), và ba lần ghép từng cặp vị trí kề nhau thành một (*patchify*, như cắt ảnh thành mảng ở Bài
  18). 24.000 mẫu mỗi giây thành 12,5 vectơ mỗi giây.
- **Bộ lượng tử hóa** RVQ có 32 lớp, mỗi lớp chọn một trong 1.024 mã, nên mỗi vectơ thành 32 số nguyên: đó là token.
- **Bộ giải mã** (decoder) đối xứng với bộ mã hóa, cũng nhân quả, dựng lại sóng âm từ token.

Mọi phần được huấn luyện cùng lúc với một tổng mất mát: mất mát mel đa tỉ lệ (mục 5), hai mất mát của bộ lượng tử
hóa (mục 6), một mất mát *ngữ nghĩa* (một mô hình ngôn ngữ 0,5 tỉ tham số đọc đầu ra của bộ lượng tử hóa và phải viết
ra lời nói hoặc mô tả âm thanh) và mất mát GAN (Bài 20). Bản thu nhỏ của ta giữ nguyên hình dạng ấy:

| | Bài báo | Ở đây |
|---|---|---|
| Dữ liệu | khoảng 3 triệu giờ tiếng nói, âm thanh, nhạc; 24 kHz | 21,9 phút chữ số tiếng Anh; 8 kHz |
| Cắt mảng | 240 · 2 · 2 · 2 = 1.920 mẫu → 12,5 khung/s | 80 · 2 · 2 · 2 = 640 mẫu → 12,5 khung/s |
| Encoder + decoder | 2 × 68 khối Transformer, rộng 768–1.280; 1,6 tỉ tham số | 2 × 8 khối, rộng 64 |
| RVQ | 32 lớp × 1.024 mã, tra cứu trong 8 chiều | giống hệt |
| Mất mát mel | cửa sổ 2⁵…2¹¹ mẫu | giống hệt (ở 8 kHz, mỗi cửa sổ dài gấp 3 về thời gian) |
| Ngữ nghĩa | LLM 0,5 tỉ tham số; nhận dạng tiếng nói, mô tả âm thanh | mô hình 2 khối đọc ra chữ số |
| Đầu ra của decoder | các mẫu sóng âm | phổ của từng khung rồi biến đổi Fourier ngược (mục 7) |

Từ các thí nghiệm của bài báo, ta tách ra sáu nhận định kiểm chứng được ở kích thước nhỏ:

| | Nhận định | Bằng chứng trong bài báo | Mục |
|---|---|---|---|
| A | *Quantizer dropout* cho một bộ token hóa dùng được ở mọi mức bit | chỉ dùng tỉ lệ 1, không so sánh | 6, 8 |
| B | Huấn luyện đầu-cuối cải thiện mãi; kiểu "partial" (đóng băng encoder và bộ lượng tử hóa) chững lại sớm | Hình 4 | 8 |
| C | Mô hình lớn hơn tốt hơn, nhất là ở mức bit cao; ở mức bit thấp, số bit mới là nút thắt | Hình 5 | 8 |
| D | Batch lớn hơn tốt hơn ở mọi thời điểm huấn luyện, cùng số bước | Hình 6 | 8 |
| E | Mất mát ngữ nghĩa làm token mang nghĩa | không có thí nghiệm bỏ nó | 8 |
| F | *Progressive Sequence Dropout* giữ TTS tốt khi bớt lớp lúc sinh; không có nó thì tụt mạnh | Hình 3 | 9 |

Bài báo không thử tỉ lệ quantizer dropout (A) và không bỏ mất mát ngữ nghĩa (E); ta thử cả hai.

## 4 · Đếm bit

Một tệp wav 8 kHz, 16 bit tốn 8.000 · 16 = 128.000 bit mỗi giây. Một token là một số trong 0…1.023, tức
log₂ 1.024 = 10 bit. Với 12,5 khung mỗi giây, mỗi lớp RVQ tốn 12,5 · 10 = 125 bit mỗi giây.

> **🤔 Dự đoán trước:** ở đủ 32 lớp, một giây tiếng nói thành bao nhiêu bit, nhỏ hơn tệp wav bao nhiêu lần? Nếu
> trải phẳng cả 32 lớp thành một chuỗi cho mô hình ngôn ngữ đọc, một giây dài bao nhiêu token?

In [ ]:
FRAME_RATE = SR / HOP
NQ, CODES = 32, 1024
BITS = math.log2(CODES)
print(f"{vn(FRAME_RATE, 1)} khung/s · {vn(BITS, 0)} bit mỗi token · {vn(FRAME_RATE * BITS, 0)} bit/s mỗi lớp")
print(" lớp |   bit/s | nhỏ hơn wav 8 kHz | nhỏ hơn wav 24 kHz (bài báo) | token/s nếu trải phẳng")
for k in (1, 2, 4, 8, 16, 32):
    bps = FRAME_RATE * k * BITS
    print(f"{k:4d} | {vn(bps, 0):>7s} | {vn(SR * 16 / bps, 0):>15s} × | {vn(24000 * 16 / bps, 0):>26s} × | "
          f"{vn(FRAME_RATE * k, 1):>8s}")  # fmt: skip
assert FRAME_RATE * NQ * BITS == 4000

Đủ 32 lớp là 4.000 bit mỗi giây (4 kbps), nhỏ hơn tệp wav 32 lần ở 8 kHz và 96 lần ở 24 kHz của bài báo. Trải
phẳng, một giây thành 400 token: một câu nói 10 giây đã là 4.000 token. Vì thế CAT-TTS (mục 9) không trải phẳng: một
Transformer chạy theo thời gian chỉ thấy 12,5 vị trí mỗi giây, và một Transformer nhỏ khác lo 32 lớp bên trong từng
khung.

Còn bao nhiêu bit là *nội dung*? Mỗi bản ghi của ta là một chữ số, tức log₂ 10 ≈ 3,3 bit, trong khoảng 0,42 giây
(trung vị): chưa đến 8 bit mỗi giây. Một lớp token (125 bit/s) đã gấp khoảng 15 lần con số đó. Phần còn lại dành cho
*giọng ai*, ngữ điệu, hơi thở, tiếng ồn: những thứ một mô hình đọc văn bản thành giọng nói cũng phải sinh ra.

## 5 · Đo khoảng cách giữa hai âm thanh

Bộ giải mã dựng lại sóng âm, rồi ta cần một hàm mất mát cho biết bản dựng lại "giống" bản gốc đến đâu. Cách thẳng
nhất là sai số bình phương trung bình (MSE) giữa từng cặp mẫu, như ở Bài 19 với từng điểm ảnh. Dưới đây là năm biến
thể của cùng chữ "seven": đảo dấu cả sóng (đổi +x thành −x), trễ 1 ms, giữ độ lớn của mọi tần số nhưng xáo pha ngẫu
nhiên, thêm nhiễu trắng 20 dB, và cắt mọi tần số trên 2 kHz.

> **🤔 Dự đoán trước:** nghe năm biến thể ở ô dưới. Xếp chúng theo mức bạn nghe thấy khác bản gốc. MSE sẽ xếp chúng
> theo thứ tự nào?

In [ ]:
x0 = torch.tensor(x_ex)[None]
g = torch.Generator().manual_seed(0)
S0 = torch.stft(x0, 256, 64, window=torch.hann_window(256), return_complex=True)
phase = torch.exp(2j * torch.pi * torch.rand(S0.shape, generator=g))
X0, f_hz = torch.fft.rfft(x0), torch.fft.rfftfreq(x0.shape[1], 1 / SR)
variants = {
    "đảo dấu": -x0,
    "trễ 1 ms": torch.roll(x0, SR // 1000, 1),
    "xáo pha": torch.istft(S0.abs() * phase, 256, 64, window=torch.hann_window(256), length=x0.shape[1]),
    "nhiễu 20 dB": x0 + torch.randn(x0.shape, generator=g) * x0.std() * 0.1,
    "lọc bỏ trên 2 kHz": torch.fft.irfft(X0 * (f_hz < 2000), n=x0.shape[1]),
}
for name, y in variants.items():
    play(y[0], name)

Đảo dấu và trễ 1 ms nghe *giống hệt* bản gốc: tai không nghe được dấu của áp suất, và 1 ms nhỏ hơn mọi khoảng thời
gian tai phân biệt được. Xáo pha làm tiếng hơi rè. Nhiễu nghe rõ là tiếng xì, còn lọc bỏ trên 2 kHz làm tiếng nghe
đục như qua điện thoại cũ. Giờ là MSE (chia cho công suất của bản gốc, nên 1 nghĩa là sai bằng cả tín hiệu):

In [ ]:
power = (x0**2).mean()
mse = {name: float(((y - x0) ** 2).mean() / power) for name, y in variants.items()}
for name, v in mse.items():
    print(f"{name:18s} MSE / công suất = {vn(v, 3)}")
assert abs(mse["đảo dấu"] - 4) < 1e-6 and min(mse, key=mse.get) == "nhiễu 20 dB"

MSE xếp ngược với tai: hai biến thể nghe giống hệt bản gốc bị phạt nặng nhất (đảo dấu sai gấp 4 lần cả tín hiệu,
vì (−x − x)² = 4x²), còn nhiễu nghe rõ lại gần như không bị phạt. MSE so từng mẫu, nên nó đòi đúng cả **pha**, tức vị
trí chính xác của từng dao động, trong khi tai chủ yếu nghe **độ lớn của từng tần số theo thời gian**. Một bộ giải
mã học bằng MSE phải đoán pha. Khi không chắc, nó trả về trung bình của các pha có thể, và trung bình của những
dao động lệch pha là gần 0: âm thanh nhỏ và mờ đi.

Vì vậy bài báo, như DAC [27] trước nó, đo khoảng cách trên **phổ mel**:

1. Biến đổi Fourier ngắn hạn (STFT): cắt sóng âm thành các cửa sổ chồng lên nhau (dài *w* mẫu, cách nhau *w*/4),
   nhân với cửa sổ Hann, lấy *độ lớn* của từng tần số. Pha bị bỏ đi.
2. Gộp các tần số thành dải theo thang **mel**: hẹp ở tần số thấp, rộng dần ở tần số cao, giống độ phân giải của
   tai.
3. Lấy log₁₀ (sau khi chặn dưới ở 10⁻⁵): tai nghe độ to theo tỉ lệ, gấp đôi năng lượng là một bước như nhau.
4. Lấy trung bình trị tuyệt đối của hiệu (L1), rồi cộng qua 7 độ dài cửa sổ *w* = 32, 64, …, 2.048 mẫu, với 5, 10,
   …, 320 dải mel.

Ta tự viết nó:

In [ ]:
MEL_WINDOWS = tuple(2**i for i in range(5, 12))  # 32 … 2.048 mẫu
MEL_BANDS = (5, 10, 20, 40, 80, 160, 320)


def hz_to_mel(f):
    return 2595 * np.log10(1 + np.asarray(f, np.float64) / 700)


def mel_filters(n_fft, n_mels):
    """Các bộ lọc tam giác trên thang mel, (n_fft/2 + 1, n_mels); mỗi bộ lọc có diện tích như nhau."""
    freqs = np.linspace(0, SR / 2, n_fft // 2 + 1)
    edges = 700 * (10 ** (np.linspace(hz_to_mel(0), hz_to_mel(SR / 2), n_mels + 2) / 2595) - 1)
    lo, mid, hi = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    tri = np.maximum(0, np.minimum((freqs[None] - lo) / (mid - lo), (hi - freqs[None]) / (hi - mid)))
    return torch.tensor((tri * (2 / (hi - lo))).T, dtype=torch.float32)


_FB = {}


def mel(x, win, n_mels):
    """|STFT| (cửa sổ Hann dài win, bước win/4) → độ lớn trong từng dải mel, (B, khung, n_mels)."""
    if (win, n_mels) not in _FB:
        _FB[win, n_mels] = (torch.hann_window(win), mel_filters(win, n_mels))
    window, fb = _FB[win, n_mels]
    s = torch.stft(x, win, win // 4, window=window, center=True, return_complex=True).abs()
    return s.transpose(1, 2) @ fb


def mel_loss(x, y):
    """Σ qua 7 độ dài cửa sổ của trung bình |log₁₀ mel(x) − log₁₀ mel(y)| (Eq. 4 của bài báo, theo cách DAC tính)."""
    total = 0
    for win, n_mels in zip(MEL_WINDOWS, MEL_BANDS, strict=True):
        a = mel(x, win, n_mels).clamp_min(1e-5).log10()
        b = mel(y, win, n_mels).clamp_min(1e-5).log10()
        total = total + (a - b).abs().mean()
    return total


mels = {name: float(mel_loss(y, x0)) for name, y in variants.items()}
print("biến thể           | MSE / công suất | mất mát mel")
for name in variants:
    print(f"{name:18s} | {vn(mse[name], 3):>15s} | {vn(mels[name], 2):>11s}")
assert mels["đảo dấu"] < 1e-6 and mels["trễ 1 ms"] < min(mels["nhiễu 20 dB"], mels["lọc bỏ trên 2 kHz"])

Mất mát mel xếp gần với tai: đảo dấu bằng đúng 0, trễ 1 ms gần 0, còn lọc bỏ trên 2 kHz bị phạt nặng nhất. Xáo pha
vẫn bị phạt khá nặng: pha ngẫu nhiên làm các cửa sổ chồng nhau triệt tiêu một phần, nên độ lớn đo lại không còn đúng
như cũ. Vậy mất mát mel cũng gián tiếp đòi pha phải *nhất quán* giữa các cửa sổ kề nhau, dù không đòi một pha cụ thể.

Vì sao cần nhiều độ dài cửa sổ? Hình dưới vẽ cùng chữ "seven" với cửa sổ ngắn nhất và dài nhất. Cửa sổ 32 mẫu
(4 ms) thấy rõ *lúc nào* âm bắt đầu và tắt, nhưng mọi tần số nhòe thành vệt dọc. Cửa sổ 2.048 mẫu (256 ms) tách rõ
từng họa âm thành vạch ngang, nhưng nhòe theo thời gian. Không cửa sổ nào thấy cả hai, nên mất mát cộng cả bảy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, win in zip(axes, (32, 2048), strict=True):
    spectrogram(ax, x_ex, f"cửa sổ {vn(win, 0)} mẫu ({vn(1000 * win / SR, 0)} ms)", win=win)
vn_axes(fig)
plt.tight_layout()
plt.show()

## 6 · Lượng tử hóa vectơ dư (RVQ)

Bộ mã hóa cho ra một vectơ thực mỗi khung. Để có token, mỗi vectơ phải được thay bằng một số nguyên: chỉ số của mã
gần nhất trong một bảng mã (*codebook*), như k-means ở chỗ mỗi điểm thuộc về tâm gần nhất. Một bảng 1.024 mã chỉ cho
10 bit mỗi khung, quá thô để dựng lại giọng nói. Muốn 40 bit, một bảng duy nhất cần 2⁴⁰ mã, hơn một nghìn tỉ.

**RVQ** làm khác: lớp 1 lượng tử hóa vectơ, lớp 2 lượng tử hóa *phần dư* (vectơ trừ mã lớp 1 đã chọn), lớp 3 lượng
tử hóa phần dư còn lại, cứ thế. Mỗi lớp chỉ cần một bảng nhỏ, bit cộng dồn, và mỗi lớp sửa lỗi của các lớp trước.
Thử với những điểm hai chiều và bảng 8 mã mỗi lớp (3 bit), tìm bằng k-means:

In [ ]:
rng = np.random.default_rng(0)
pts = np.concatenate([rng.normal(c, 0.35, (400, 2)) for c in rng.uniform(-3, 3, (6, 2))])


def kmeans(P, k, seed, iters=30):
    r = np.random.default_rng(seed)
    c = P[r.choice(len(P), k, replace=False)]
    for _ in range(iters):
        a = ((P[:, None] - c[None]) ** 2).sum(-1).argmin(1)
        c = np.array([P[a == j].mean(0) if (a == j).any() else c[j] for j in range(k)])
    return c, a


res, rec, books, errs = pts.copy(), np.zeros_like(pts), [], []
for layer in range(6):
    c, a = kmeans(res, 8, seed=layer)
    books.append((res.copy(), c))
    rec += c[a]
    res = pts - rec
    errs.append(float((res**2).sum(1).mean()))
for layer, e in enumerate(errs, 1):
    print(f"{layer} lớp = {3 * layer:2d} bit: sai số bình phương trung bình {vn(e, 4)}")
assert all(np.diff(errs) < 0)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, (r, c), name in zip(axes[:2], books[:2], ("lớp 1: chính các điểm", "lớp 2: phần dư sau lớp 1"), strict=True):
    ax.scatter(r[:, 0], r[:, 1], s=3, color=BLUE, alpha=0.4, label="điểm cần lượng tử hóa")
    ax.scatter(c[:, 0], c[:, 1], s=60, marker="s", color=ORANGE, edgecolor="k", label="8 mã của lớp")
    ax.set(title=name, aspect="equal")
axes[0].legend(loc="lower left", fontsize=8)
axes[2].plot(range(3, 19, 3), errs, "o-", color=GREEN)
axes[2].set(yscale="log", xlabel="tổng số bit", ylabel="sai số bình phương trung bình", title="mỗi lớp thêm 3 bit")
vn_axes(fig)
plt.tight_layout()
plt.show()

Hình giữa có cùng thang đo với hình trái: phần dư sau lớp 1 đã nhỏ hơn nhiều, và lớp 2 đặt 8 mã của nó quanh gốc
tọa độ, nơi phần dư tập trung. Mỗi lớp giảm sai số vài lần. Thứ tự này quan trọng: **các lớp đầu mang phần thô, các
lớp sau tinh chỉnh**. Cắt bớt lớp cuối thì còn một bản xấp xỉ thô hơn chứ không hỏng hẳn. Đó là thứ bài báo dùng để
chọn mức bit lúc dùng: giữ K lớp đầu là 125·K bit mỗi giây.

### Một lớp RVQ của CAT

CAT không chạy k-means. Mỗi lớp là một mô-đun học bằng gradient, theo DAC [27]:

- **Phân tích thừa số** (*factorized*): chiếu phần dư 64 chiều xuống 8 chiều, tìm mã gần nhất trong 8 chiều, rồi
  chiếu mã ấy ngược lên 64 chiều. Tìm trong không gian nhỏ thì nhiều mã được dùng hơn.
- **Chuẩn hóa L2**: "gần nhất" tính bằng cosin, tức sau khi đưa cả vectơ lẫn mã về độ dài 1. Thứ được chiếu ngược
  là mã chưa chuẩn hóa.
- Chọn mã (argmax) không có đạo hàm. **Straight-through**: lúc lan truyền ngược, coi như lượng tử hóa là hàm đồng
  nhất, tức gradient ở đầu ra được chuyển nguyên về đầu vào. Trong code là `ze + (zq − ze).detach()`: giá trị bằng
  `zq`, còn gradient theo `ze`.
- Vì straight-through bỏ qua bảng mã, bảng mã cần mất mát riêng. Hai mất mát của bài báo (Eq. 2–3), với sg là dừng
  gradient: ℒ_cmt = ‖z − sg(q(z))‖² kéo đầu ra của bộ mã hóa về phía mã đã chọn, ℒ_code = ‖sg(z) − q(z)‖² kéo mã về
  phía đầu ra của bộ mã hóa. Trọng số 0,25 và 1, như VQ-VAE và như bài báo.

In [ ]:
class VQLayer(nn.Module):
    """Một lớp RVQ kiểu DAC: chiếu d → cdim, tìm mã gần nhất theo cosin, chiếu mã (chưa chuẩn hóa) về lại d."""

    def __init__(self, d, codes=CODES, cdim=8):
        super().__init__()
        self.inp, self.outp = nn.Linear(d, cdim), nn.Linear(cdim, d)
        self.book = nn.Parameter(torch.randn(codes, cdim))

    def forward(self, r):
        ze = self.inp(r)
        idx = (F.normalize(ze.detach(), dim=-1) @ F.normalize(self.book, dim=-1).T).argmax(-1)
        zq = self.book[idx]
        cmt = ((ze - zq.detach()) ** 2).mean(-1)  # Eq. 2, cho từng vectơ
        cb = ((zq - ze.detach()) ** 2).mean(-1)  # Eq. 3
        return self.outp(ze + (zq - ze).detach()), idx, cmt, cb


torch.manual_seed(0)
layer = VQLayer(64)
r = torch.randn(64)
q, code, cmt, cb = layer(r[None])
print(
    f"vectơ 64 chiều → token {int(code[0])} (10 bit) → vectơ 64 chiều; ℒ_cmt = {vn(cmt.item(), 3)}, ℒ_code = {vn(cb.item(), 3)}"
)
J = torch.autograd.functional.jacobian(lambda v: layer(v[None])[0][0], r)
assert torch.allclose(J, layer.outp.weight @ layer.inp.weight, atol=1e-6)
print("Jacobian của đầu ra theo đầu vào = W_out · W_in: gradient đi qua argmax như thể không có lượng tử hóa")

### Quantizer dropout: học để bị cắt bớt

Ở ví dụ k-means, mỗi lớp được tìm sau khi các lớp trước đã cố định, nên lớp đầu tự nhiên mang phần thô. Khi bộ mã
hóa, các lớp và bộ giải mã được học *cùng lúc* với đủ 8 lớp, không gì bắt các lớp đầu phải tự đứng được: mạng chỉ
cần tổng của cả 8 lớp đúng. **Quantizer dropout** (SoundStream, 2021) sửa điều đó: với mỗi ví dụ huấn luyện, chọn ngẫu
nhiên số lớp K từ 1 đến 8 và chỉ dùng K lớp đầu. Bài báo dùng nó cho *mọi* ví dụ (tỉ lệ 1), như SoundStream. DAC
chỉ dùng cho một nửa số ví dụ, vì dùng cho mọi ví dụ làm giảm chất lượng khi giữ đủ số lớp.

Thử trên một bài toán nhỏ: nén mỗi khung phổ mel 40 dải của FSDD (10 ms, chỉ lấy khung có tiếng) qua một bộ mã hóa
MLP, 8 lớp `VQLayer` với 16 mã mỗi lớp (4 bit), rồi một bộ giải mã MLP. Hai mô hình giống hệt nhau, một học với đủ 8
lớp, một học với số lớp ngẫu nhiên.

> **🤔 Dự đoán trước:** khi chỉ giữ 1 hoặc 2 lớp đầu, mô hình nào sai ít hơn? Còn khi giữ đủ 8 lớp?

In [ ]:
def logmel(x):
    """(B, L) → (B, 40, khung): log₁₀ độ lớn trong 40 dải mel; cửa sổ 32 ms, bước 10 ms."""
    if "logmel" not in _FB:
        _FB["logmel"] = (torch.hann_window(256), mel_filters(256, 40))
    window, fb = _FB["logmel"]
    s = torch.stft(x, 256, 80, window=window, center=True, return_complex=True).abs()
    return (s.transpose(1, 2) @ fb).clamp_min(1e-5).log10().transpose(1, 2)


def pad_clips(ids):
    out = np.zeros((len(ids), WIN * HOP), np.float32)
    for b, c in enumerate(ids):
        out[b, : len(waves[c])] = waves[c]
    return torch.tensor(out)


def mel_frames(ids):
    m = logmel(pad_clips(ids)).transpose(1, 2).reshape(-1, 40)
    return m[m.max(1).values > -3]  # bỏ các khung im lặng


F_TR, F_VA = mel_frames(TRAIN[:600]), mel_frames(VAL)
mu, sd = F_TR.mean(0), F_TR.std(0)
F_TR, F_VA = (F_TR - mu) / sd, (F_VA - mu) / sd


class ToyRVQ(nn.Module):
    def __init__(self, nq=8, codes=16, d=32, cdim=4):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(40, 128), nn.GELU(), nn.Linear(128, d))
        self.dec = nn.Sequential(nn.Linear(d, 128), nn.GELU(), nn.Linear(128, 40))
        self.layers = nn.ModuleList(VQLayer(d, codes, cdim) for _ in range(nq))

    def forward(self, x, n_q):
        r = self.enc(x)
        out, vq = torch.zeros_like(r), 0
        for i, lay in enumerate(self.layers):
            m = (i < n_q).float()  # ví dụ nào dùng ít lớp hơn thì lớp này bằng 0
            q, _, cmt, cb = lay(r)
            q = q * m[:, None]
            out, r = out + q, r - q
            vq = vq + ((0.25 * cmt + cb) * m).sum() / m.sum()  # mất mát chỉ tính trên các ví dụ dùng lớp này
        return self.dec(out), vq


def train_toy(qdrop, seed=0, steps=300, bs=256, nq=8):
    torch.manual_seed(seed)
    m = ToyRVQ(nq)
    opt = torch.optim.Adam(m.parameters(), 3e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        b = F_TR[torch.randint(len(F_TR), (bs,), generator=g)]
        drop = torch.rand(bs, generator=g) < qdrop
        n_q = torch.where(drop, torch.randint(1, nq + 1, (bs,), generator=g), torch.tensor(nq))
        y, vq = m(b, n_q)
        loss = F.mse_loss(y, b) + vq
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        return [F.mse_loss(m(F_VA, torch.full((len(F_VA),), k))[0], F_VA).item() for k in range(1, nq + 1)]


toy = {qd: train_toy(qd) for qd in (0.0, 1.0)}
LABEL = {0.0: "học với đủ 8 lớp", 1.0: "học với số lớp ngẫu nhiên"}
print("số lớp giữ lại".ljust(26) + "".join(f"{k:>6d}" for k in range(1, 9)))
for qd, errs_k in toy.items():
    print(LABEL[qd].ljust(26) + "".join(f"{vn(e, 2):>6s}" for e in errs_k))
assert toy[1.0][0] < toy[0.0][0] and toy[1.0][1] < toy[0.0][1] and toy[0.0][-1] < toy[1.0][-1]

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(range(1, 9), toy[0.0], "o-", color=ORANGE, label=LABEL[0.0])
ax.plot(range(1, 9), toy[1.0], "s--", color=BLUE, label=LABEL[1.0])
ax.set(xlabel="số lớp giữ lại lúc dùng", ylabel="MSE trên tập kiểm định", title="quantizer dropout, bài toán nhỏ")
ax.legend()
vn_axes(fig)
plt.tight_layout()
plt.show()

Mô hình chỉ học với đủ 8 lớp sai nặng khi bị cắt bớt (đường cam): các lớp đầu của nó không được dạy để tự đứng, và
thêm một lớp đôi khi còn làm kết quả *tệ hơn*. Mô hình học với số lớp ngẫu nhiên (đường xanh) sai ít hơn nhiều ở 1–2
lớp. Đổi lại, ở đủ 8 lớp nó sai *nhiều hơn*, và đường của nó gần như nằm ngang từ lớp 4: lớp thứ k chỉ được huấn
luyện ở những ví dụ giữ ít nhất k lớp, tức (9 − k)/8 số ví dụ, nên các lớp cuối học ít nhất. Đó là lý do DAC chỉ áp
dụng dropout cho một nửa số ví dụ. Mục 8 đo sự đánh đổi này trên chính bộ token hóa âm thanh, với tỉ lệ 0, ½ và 1.

## 7 · CAT thu nhỏ

Giờ ráp mọi thứ thành bộ token hóa. Bộ mã hóa nhận một đoạn 1,28 giây (10.240 mẫu):

1. Cắt thành 128 mảng 80 mẫu (10 ms mỗi mảng), như cắt ảnh thành mảng ở ViT.
2. **Giai đoạn 1**: chiếu mỗi mảng lên 64 chiều, qua 2 khối Transformer nhân quả, chiếu xuống 32 chiều.
3. **Patchify**: ghép từng cặp vị trí kề nhau thành một vectơ 64 chiều, nên 128 vị trí còn 64 (20 ms mỗi vị trí).
4. Lặp lại thêm hai lần (giai đoạn 2, 3): 64 → 32 → 16 vị trí. Giai đoạn 4 cho ra 16 vectơ 64 chiều, mỗi vectơ cho
   80 ms: đúng 12,5 khung mỗi giây.
5. RVQ 32 lớp biến mỗi vectơ thành 32 token.

Bộ giải mã đi ngược lại: cộng các mã đã chọn, qua 4 giai đoạn, mỗi lần tách một vị trí thành hai (*unpatchify*).
Mỗi khối dùng chú ý nhân quả (vị trí t chỉ nhìn các vị trí ≤ t), mã hóa vị trí bằng RoPE như các LLM hiện nay, và
nhân mỗi nhánh với một *layer scale* học được, khởi tạo 0,01, như bài báo, để mạng sâu bắt đầu gần như hàm đồng
nhất.

**Một chỗ khác bài báo.** CAT của bài báo cho ra trực tiếp 1.920 mẫu sóng âm mỗi khung. Bản của ta cho ra, mỗi 10 ms,
*phổ* của một cửa sổ 40 ms (log độ lớn và pha của 161 tần số), rồi biến đổi Fourier ngược và cộng chồng các cửa sổ,
như Vocos (Siuzdak, 2023). Trong các lần chạy thử ngắn (3.000 bước), cách cho ra từng mẫu học chậm hơn hẳn ở kích
thước này; mục 8 so sánh hai cách trên cùng ngân sách 10.000 bước. Đầu ra vẫn do Transformer sinh ra và không có lớp
tích chập nào, nhưng ta trả một giá: cửa sổ 40 ms trải sang 30 ms của khung sau, nên 30 ms cuối của mỗi khung chỉ xong
khi token của khung kế tiếp đến. Bài báo không có độ trễ này.

In [ ]:
PATCH = (80, 2, 2, 2)


def rope(t, dh):
    inv = 1.0 / (10000 ** (torch.arange(0, dh, 2, dtype=torch.float32) / dh))
    ang = torch.arange(t, dtype=torch.float32)[:, None] * inv[None]
    return ang.cos(), ang.sin()


def apply_rope(x, cs):
    cos, sin = cs
    a, b = x[..., 0::2], x[..., 1::2]
    return torch.stack((a * cos - b * sin, a * sin + b * cos), -1).flatten(-2)


class Block(nn.Module):
    """Khối Transformer pre-LN: chú ý nhân quả (RoPE) rồi MLP GELU 4×, mỗi nhánh nhân với layer scale (khởi tạo 0,01)."""

    def __init__(self, d, heads=4, ls=0.01):
        super().__init__()
        self.h = heads
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        self.f1, self.f2 = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.s1 = nn.Parameter(torch.full((d,), ls))
        self.s2 = nn.Parameter(torch.full((d,), ls))

    def forward(self, x, cs):
        B, T, d = x.shape
        q, k, v = self.qkv(self.n1(x)).view(B, T, 3, self.h, d // self.h).permute(2, 0, 3, 1, 4)
        q, k = apply_rope(q, cs), apply_rope(k, cs)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.s1 * self.o(a.transpose(1, 2).reshape(B, T, d))
        return x + self.s2 * self.f2(F.gelu(self.f1(self.n2(x))))


class Stage(nn.Module):
    """Chiếu vào → các khối nhân quả → chiếu ra (ProjectedTransformer của MOSS)."""

    def __init__(self, d_in, d, layers, d_out, heads=4):
        super().__init__()
        self.inp, self.out = nn.Linear(d_in, d), nn.Linear(d, d_out)
        self.blocks = nn.ModuleList(Block(d, heads) for _ in range(layers))
        self.dh = d // heads

    def forward(self, x):
        cs = rope(x.shape[1], self.dh)
        x = self.inp(x)
        for b in self.blocks:
            x = b(x, cs)
        return self.out(x)


class RVQ(nn.Module):
    """32 lớp VQLayer nối tiếp trên phần dư."""

    def __init__(self, d, nq=NQ):
        super().__init__()
        self.layers = nn.ModuleList(VQLayer(d) for _ in range(nq))

    def encode(self, z, k):
        res, codes = z, []
        for lay in self.layers[:k]:
            q, idx, _, _ = lay(res)
            res = res - q
            codes.append(idx)
        return torch.stack(codes, -1)

    def decode(self, codes):
        return sum(lay.outp(lay.book[codes[..., i]]) for i, lay in enumerate(self.layers[: codes.shape[-1]]))


def overlap_add(frames, hop):
    """(B, T, n) → (B, T·hop): cửa sổ t kết thúc ở mẫu (t + 1)·hop; phần trước thời điểm 0 bị bỏ."""
    B, T, n = frames.shape
    y = F.fold(frames.transpose(1, 2), (1, (T - 1) * hop + n), (1, n), stride=(1, hop))
    return y.reshape(B, -1)[:, n - hop : n - hop + T * hop]


TAG, BOS_T, EOS_T = 10, 11, 12  # từ vựng văn bản: chữ số 0–9, nhãn nhiệm vụ, bắt đầu, kết thúc


class SemanticLM(nn.Module):
    """Mục tiêu ngữ nghĩa thu nhỏ: [nhãn nhiệm vụ] + đầu ra của RVQ + lời nói; nhân quả, đoán từng chữ số."""

    def __init__(self, d_audio, d=64, layers=2):
        super().__init__()
        self.audio = nn.Linear(d_audio, d)
        self.emb = nn.Embedding(13, d)
        self.blocks = nn.ModuleList(Block(d) for _ in range(layers))
        self.norm, self.head = nn.LayerNorm(d), nn.Linear(d, 13)
        self.dh = d // 4

    def logits(self, q, text_in):
        B, T = q.shape[:2]
        h = torch.cat([self.emb(torch.full((B, 1), TAG)), self.audio(q), self.emb(text_in)], 1)
        cs = rope(h.shape[1], self.dh)
        for b in self.blocks:
            h = b(h, cs)
        return self.head(self.norm(h[:, 1 + T :]))

    @torch.no_grad()
    def transcribe(self, q, max_len=4):
        """Đọc tham lam (greedy) đến khi gặp token kết thúc."""
        tin, out = torch.full((q.shape[0], 1), BOS_T), [[] for _ in range(q.shape[0])]
        done = torch.zeros(q.shape[0], dtype=torch.bool)
        for _ in range(max_len):
            nxt = self.logits(q, tin)[:, -1].argmax(-1)
            for b in range(q.shape[0]):
                if not done[b]:
                    done[b] = bool(nxt[b] > 9)
                    if not done[b]:
                        out[b].append(int(nxt[b]))
            tin = torch.cat([tin, nxt[:, None]], 1)
        return out


class CAT(nn.Module):
    def __init__(self, d=64, layers=(2, 2, 2, 2)):
        super().__init__()
        p0 = PATCH[0]
        enc = [(p0, d // 2), (d, d // 2), (d, d // 2), (d, d)]  # (vào, ra) của từng giai đoạn
        dec = [(d, d), (d // 2, d), (d // 2, d), (d // 2, 2 * (2 * p0 + 1))]
        self.enc = nn.ModuleList(Stage(i, d, n, o) for (i, o), n in zip(enc, layers, strict=True))
        self.dec = nn.ModuleList(Stage(i, d, n, o) for (i, o), n in zip(dec, layers[::-1], strict=True))
        self.rvq = RVQ(d)
        self.sem = SemanticLM(d)

    def encode_z(self, x):
        h = x.view(x.shape[0], -1, PATCH[0])  # (B, 128, 80) cho một đoạn 1,28 s
        for s, st in enumerate(self.enc):
            if s:
                h = h.reshape(h.shape[0], h.shape[1] // 2, -1)  # patchify: ghép 2 vị trí kề nhau
            h = st(h)
        return h  # (B, 16, 64): 12,5 vectơ mỗi giây

    def decode_q(self, q):
        h = q
        for s, st in enumerate(self.dec):
            h = st(h)
            if s < len(self.dec) - 1:
                h = h.reshape(h.shape[0], h.shape[1] * 2, -1)  # unpatchify: tách 1 vị trí thành 2
        p0 = PATCH[0]
        n = 2 * p0 + 1  # 161 tần số của một cửa sổ 320 mẫu
        mag = h[..., :n].clamp(max=math.log(100)).exp()
        frames = torch.fft.irfft(torch.polar(mag, h[..., n:]), 4 * p0) * torch.hann_window(4 * p0) / 1.5
        return overlap_add(frames, p0)  # mỗi 10 ms một cửa sổ 40 ms

    @torch.no_grad()
    def encode(self, x, k=NQ):
        return self.rvq.encode(self.encode_z(x), k)

    @torch.no_grad()
    def decode(self, codes):
        return self.decode_q(self.rvq.decode(codes))

Huấn luyện bộ token hóa này mất hơn một giờ trên 4 nhân CPU, quá lâu cho một notebook. Script `reel/codec/sweep.py`
đã huấn luyện nó (10.000 bước, batch 32 đoạn 1,28 giây, AdamW, η = 3 · 10⁻³ chọn trong {10⁻³; 3 · 10⁻³; 10⁻²} trên
tập kiểm định) với hàm mất mát của bài báo, cùng trọng số: 15 · ℒ_mel + 0,25 · ℒ_cmt + 1 · ℒ_code + 20 · ℒ_sem, và
quantizer dropout cho mọi ví dụ. Ta tải bản seed 0, lưu ở dạng int8 (mỗi hàng một hệ số tỉ lệ); mọi con số của
script cũng tính trên đúng các trọng số int8 này.

In [ ]:
R = fetch_json("codec-results.json", "53c34e984a6054d6135be5afd7c9128d")
W = fetch_json("codec-weights.json", "6f4b0e5508a869f770f628dd0468e742")


def unpack(p):
    """Giải nén trọng số: int8 × hệ số tỉ lệ float16 của từng hàng, hoặc float16."""
    sd = {}
    for k, v in p.items():
        if "q" in v:
            q = np.frombuffer(base64.b64decode(v["q"]), np.int8).astype(np.float32).reshape(v["shape"][0], -1)
            s = np.frombuffer(base64.b64decode(v["s"]), "<f2").astype(np.float32)
            sd[k] = torch.tensor((q * s[:, None]).reshape(v["shape"]))
        else:
            raw = np.frombuffer(base64.b64decode(v["f16"]), "<f2").astype(np.float32)
            sd[k] = torch.tensor(raw.reshape(v["shape"]))
    return sd


codec = CAT()
codec.load_state_dict(unpack(W["codec"]))
codec.eval()
n_codec = sum(p.numel() for n, p in codec.named_parameters() if not n.startswith("sem."))
n_sem = sum(p.numel() for p in codec.sem.parameters())
print(f"CAT thu nhỏ: {vn(n_codec, 0)} tham số, cộng {vn(n_sem, 0)} của mô hình ngữ nghĩa; bài báo: 1,6 tỉ")
assert n_codec == R["codec"]["ref"][0]["params"] and W["cfg"]["steps"] == R["steps"] == 10_000
assert R["eta"]["chosen"] == 3e-3 and R["eta"]["grid"] == [1e-3, 3e-3, 1e-2]

x1 = pad_clips([EX])
h = x1.view(1, -1, PATCH[0])
print(f"{'đoạn 1,28 s cắt thành mảng 80 mẫu':38s} {tuple(h.shape)}")
with torch.no_grad():
    for s, st in enumerate(codec.enc):
        if s:
            h = h.reshape(1, h.shape[1] // 2, -1)
            print(f"{'  patchify (ghép 2 vị trí)':38s} {tuple(h.shape)}")
        h = st(h)
        print(f"{f'giai đoạn {s + 1} (2 khối, rộng 64)':38s} {tuple(h.shape)}")
    codes_ex = codec.rvq.encode(h, NQ)
print(f"{'RVQ: 32 token cho mỗi khung 80 ms':38s} {tuple(codes_ex.shape)}")
print("32 token của khung thứ 5:", codes_ex[0, 4].tolist())

**Nhân quả, nên chạy được theo luồng.** Vì mọi khối chỉ nhìn về quá khứ, vectơ của T khung đầu phải giống hệt nhau
dù ta đưa vào T · 80 ms đầu hay cả 1,28 giây: bộ mã hóa không cần chờ câu nói kết thúc. Ở chiều giải mã, đầu ra
đến 30 ms trước cuối khung cuối cùng cũng phải không đổi; 30 ms còn lại chờ khung kế tiếp (cái giá của đầu ra dạng
phổ ở trên).

In [ ]:
with torch.no_grad():
    z_full, codes_full = codec.encode_z(x1), codec.encode(x1)
    y_full = codec.decode(codes_full)
    for T in (2, 3, 4, 5):
        assert torch.allclose(codec.encode_z(x1[:, : T * HOP]), z_full[:, :T], atol=1e-5)
        y_T = codec.decode(codes_full[:, :T])
        done = T * HOP - 3 * PATCH[0]
        assert torch.allclose(y_T[:, :done], y_full[:, :done], atol=1e-5)
        tail = float((y_T[:, done:] - y_full[:, done : T * HOP]).abs().max())
        print(f"{T} khung đầu: vectơ của bộ mã hóa giống hệt bản đủ 1,28 s; âm thanh giống hệt đến "
              f"{vn(1000 * done / SR, 0)} ms, 30 ms cuối lệch tới {vn(tail, 3)}")  # fmt: skip

### Nghe chữ số đi qua 1 … 32 lớp

Cùng chữ "seven" ở mục 2, mã hóa một lần thành 16 × 32 token, rồi giải mã chỉ từ K lớp đầu. Hãy nghe từ 1 lớp
(125 bit/s) lên 32 lớp (4.000 bit/s), và để ý lúc nào bạn nhận ra chữ số, lúc nào nhận ra giọng người đọc.

In [ ]:
with torch.no_grad():
    recon = {k: codec.decode(codes_full[..., :k])[0, : len(x_ex)].numpy() for k in (1, 2, 4, 8, 16, 32)}
play(x_ex, "bản gốc (128.000 bit/s)")
for k, y in recon.items():
    play(y, f"{k} lớp = {vn(FRAME_RATE * k * BITS, 0)} bit/s")

fig, axes = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
spectrogram(axes[0], x_ex, "bản gốc")
for ax, k in zip(axes[1:], (1, 4, 32), strict=True):
    spectrogram(ax, recon[k], f"{k} lớp ({vn(FRAME_RATE * k * BITS, 0)} bit/s)")
for ax in axes[1:]:
    ax.set_ylabel("")
vn_axes(fig)
plt.tight_layout()
plt.show()

### Đo bằng hai "người chấm"

Bài báo đo bằng các mô hình có sẵn: lỗi từ (WER) của một mô hình nhận dạng tiếng nói, và độ giống giọng (SIM) là
cosin giữa embedding của một mô hình xác minh người nói cho bản gốc và bản dựng lại. Ta dựng hai người chấm nhỏ thay
thế, huấn luyện trên các bản ghi huấn luyện *gốc* (không phải bản đã qua bộ mã hóa): một CNN đoán chữ số, một CNN đoán
người đọc. Embedding 128 chiều của CNN thứ hai cho SIM. Một nửa số lô huấn luyện của chúng bị làm hỏng ngẫu nhiên
(nhiễu, lọc thấp, nghiêng phổ), để chúng "nghe xuyên" qua méo tiếng như một mô hình thật, thay vì sụp vì những khác
biệt nhỏ. Trên các bản ghi kiểm tra gốc, chúng đúng như sau:

In [ ]:
class Judge(nn.Module):
    """CNN 1-D trên 40 dải log-mel; gộp trung bình + cực đại theo thời gian → embedding 128 chiều → lớp."""

    def __init__(self, classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(40, 128, 5, padding=2), nn.ReLU(),
            nn.Conv1d(128, 128, 5, stride=2, padding=2), nn.ReLU(),
            nn.Conv1d(128, 128, 5, stride=2, padding=2), nn.ReLU(),
        )  # fmt: skip
        self.emb, self.head = nn.Linear(256, 128), nn.Linear(128, classes)

    def embed(self, x):
        h = self.conv((logmel(x) + 5) / 2)
        return F.relu(self.emb(torch.cat([h.mean(-1), h.amax(-1)], 1)))

    def forward(self, x):
        return self.head(self.embed(x))


jd, js = Judge(10), Judge(len(SPEAKERS))
jd.load_state_dict(unpack(W["judgeDigit"]))
js.load_state_dict(unpack(W["judgeSpk"]))
jd.eval(), js.eval()
x_test = pad_clips(TEST)
with torch.no_grad():
    acc_d = float((jd(x_test).argmax(-1).numpy() == digit[TEST]).mean())
    acc_s = float((js(x_test).argmax(-1).numpy() == spk[TEST]).mean())
print(f"người chấm chữ số: {pct(acc_d)} · người chấm giọng: {pct(acc_s)} (300 bản ghi kiểm tra gốc)")
assert abs(acc_d - R["judges"]["digit"]["test"]) <= 1 / 300 and abs(acc_s - R["judges"]["spk"]["test"]) <= 1 / 300

### Bộ token hóa trên tập kiểm tra

> **🤔 Dự đoán trước:** người chấm cần bao nhiêu lớp token để nhận ra *chữ số*? Bao nhiêu để nhận ra *giọng*? Cái
> nào cần nhiều bit hơn?

Ô dưới chạy trực tiếp bộ token hóa seed 0 trên 60 bản ghi kiểm tra (mỗi người mỗi chữ số một bản), ở 1, 4 và 32 lớp,
rồi so với kết quả của script cho chính những bản ghi ấy. Bảng sau đó là cả 300 bản ghi, trung bình 3 seed. Ngoài
hai người chấm, mô hình ngữ nghĩa của chính bộ token hóa cũng "nghe" được: nó đọc token và viết ra chữ số (cột
*ASR từ token*).

In [ ]:
PC = W["perClip"]
pos = {c: i for i, c in enumerate(PC["ids"])}
SUB = [c for c in TEST if idx[c] == 0]
x_sub = pad_clips(SUB)
f_sub = np.array([-(-n_samples[c] // HOP) for c in SUB])
mask = (torch.arange(WIN * HOP)[None] < torch.tensor(f_sub * HOP)[:, None]).float()
with torch.no_grad():
    c_sub = codec.encode(x_sub)
    for k in (1, 4, 32):
        y = codec.decode(c_sub[..., :k]) * mask
        live = {
            "mel": [float(mel_loss(y[i : i + 1, : f * HOP], x_sub[i : i + 1, : f * HOP])) for i, f in enumerate(f_sub)],
            "digit": (jd(y).argmax(-1).numpy() == digit[SUB]).astype(int),
            "spk": (js(y).argmax(-1).numpy() == spk[SUB]).astype(int),
            "sim": F.cosine_similarity(js.embed(y), js.embed(x_sub)).numpy(),
            "asr": [int(t == [d]) for t, d in zip(codec.sem.transcribe(codec.rvq.decode(c_sub[..., :k])), digit[SUB])],
        }
        ki = PC["k"].index(k)
        ref_ = {key: np.array([PC[key][ki][pos[c]] for c in SUB]) for key in live}
        same = {key: float(np.mean(np.isclose(live[key], ref_[key], atol=2e-3))) for key in live}
        print(f"{k:2d} lớp: mel {vn(np.mean(live['mel']), 3)} · chữ số {pct(np.mean(live['digit']))} · giọng "
              f"{pct(np.mean(live['spk']))} · SIM {vn(np.mean(live['sim']), 3)} · ASR từ token {pct(np.mean(live['asr']))}"
              f"; khớp với script ở {pct(min(same.values()), 0)} số bản ghi")  # fmt: skip
        assert min(same.values()) >= 0.95


def seeds(name, split="test"):
    return {key: np.array([r[split][key] for r in R["codec"][name]]) for key in ("mel", "asr", "digit", "spk", "sim")}


ref = seeds("ref")
print("\n lớp |  bit/s | mel  | ASR từ token | chữ số | giọng | SIM   (300 bản ghi kiểm tra, trung bình 3 seed)")
for i, k in enumerate(R["ks"]):
    print(f"{k:4d} | {vn(FRAME_RATE * k * BITS, 0):>6s} | {vn(ref['mel'][:, i].mean(), 2)} | "
          f"{pct(ref['asr'][:, i].mean()):>12s} | {pct(ref['digit'][:, i].mean()):>6s} | {pct(ref['spk'][:, i].mean()):>5s} "
          f"| {vn(ref['sim'][:, i].mean(), 3)}")  # fmt: skip

Đọc bảng từ trên xuống:

- **Nội dung rẻ, giọng đắt.** Chỉ 1 lớp (125 bit/s), người chấm đã nghe đúng chữ số 81,9% số lần, còn giọng thì chỉ
  72,0%. Giọng cần khoảng 4 lớp (500 bit/s) mới lên 90,6%. Đúng như phép đếm ở mục 4: nội dung chỉ cần vài bit mỗi
  giây, phần lớn số bit còn lại là để tả giọng ai đang nói.
- **Token biết nhiều hơn âm thanh dựng lại.** Mô hình ngữ nghĩa đọc thẳng token 1 lớp và viết đúng chữ số 92,4% số lần,
  trong khi người chấm nghe âm thanh giải mã từ cùng token ấy chỉ đúng 81,9%. Thông tin có trong token; bộ giải mã nhỏ
  chưa dựng lại hết. (Người chấm học trên bản ghi gốc, còn mô hình ngữ nghĩa học trên chính các token này, nên so sánh
  này nghiêng về mô hình ngữ nghĩa.)
- **Từ khoảng 8 lớp trở đi, thêm bit gần như không giúp gì:** khoảng cách mel 1,62 ở 4 lớp, 1,59 ở 8 lớp và vẫn 1,59 ở
  32 lớp, cách khá xa mức người chấm đạt trên bản gốc (97,7% chữ số, 96,3% giọng). Ở kích thước này, nút thắt là mô
  hình chứ không phải số bit. Phổ đồ ở trên cho thấy điều đó: ngay cả ở 32 lớp, chỉ một hai họa âm thấp nhất còn rõ,
  các họa âm cao hơn nhòe thành một dải rộng. Mục 8 (C) thử một mô hình rộng hơn.

Một cách khác để thấy các lớp cuối mang ít thông tin: đếm xem mỗi lớp thật sự dùng bao nhiêu mã. Nếu một lớp dùng
đều cả 1.024 mã thì mỗi token của nó đáng 10 bit; nếu nó gần như luôn chọn vài mã quen thuộc thì ít hơn nhiều. Entropy
của tần suất các mã, tính bằng bit, là số bit mà một bộ nén lý tưởng cần cho mỗi token của lớp đó (chưa tính quan hệ
giữa các lớp). Script đã đếm các mã của bộ token hóa seed 0 trên mọi khung của tập huấn luyện.

In [ ]:
U = PC["usage"]
eff = np.log2(U["perplexity"])  # perplexity = 2^entropy, nên log₂ của nó là entropy tính bằng bit
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, NQ + 1), eff, color=BLUE, label="entropy của tần suất các mã")
ax.axhline(BITS, color=GREY, ls="--", lw=1, label="tối đa 10 bit: cả 1.024 mã được dùng đều nhau")
ax.set(xlabel="lớp RVQ", ylabel="bit mỗi token", title=f"{vn(U['frames'], 0)} khung của tập huấn luyện, seed 0")
ax.set_ylim(0, 12.5)
ax.legend(fontsize=8, loc="upper right", ncol=2, frameon=False)
vn_axes(fig)
plt.tight_layout()
plt.show()
print(f"lớp 1 dùng {vn(U['distinct'][0], 0)} mã khác nhau, lớp 8 dùng {vn(U['distinct'][7], 0)}, lớp 32 chỉ "
      f"{vn(U['distinct'][-1], 0)}")  # fmt: skip
print(f"tổng: {vn(eff.sum(), 1)} bit mỗi khung thay vì {vn(NQ * BITS, 0)}, tức khoảng "
      f"{vn(eff.sum() * FRAME_RATE, 0)} bit/s thay vì 4.000")  # fmt: skip
assert eff[-4:].max() < eff[1:16].min() and U["frames"] == sum(-(-n_samples[c] // HOP) for c in TRAIN)

Lớp 2 đến khoảng lớp 17 dùng gần hết bảng mã (9,2–9,8 bit mỗi token), còn các lớp cuối chỉ dùng từ vài trăm mã
xuống còn 18. Một phần lý do có lẽ là quantizer dropout ở mục 6: lớp thứ k chỉ được học ở những ví dụ giữ ít nhất k
lớp, tức (33 − k)/32 số ví dụ, nên lớp 32 chỉ học ở 1/32 số ví dụ. Lớp 1 cũng không dùng hết bảng mã (8,1 bit). Vậy
danh nghĩa là 4.000 bit/s, nhưng một bộ nén tốt chỉ cần khoảng 3.300.

## 8 · Kiểm chứng các nhận định

Mỗi cấu hình dưới đây là bộ token hóa ở mục 7 với đúng *một* thay đổi, cùng 10.000 bước, cùng η và 3 seed. Đường là
trung bình 3 seed, dải mờ là từ seed thấp nhất đến seed cao nhất. Mọi số đo là trên 300 bản ghi kiểm tra.

In [ ]:
BPS = FRAME_RATE * np.array(R["ks"]) * BITS
STY = [(BLUE, "o", "-"), (ORANGE, "s", "--"), (GREEN, "^", ":"), (GREY, "D", "-.")]
YL = {"mel": "khoảng cách mel (thấp là tốt)", "digit": "người chấm đúng chữ số (%)", "spk": "người chấm đúng giọng (%)",
      "sim": "SIM", "asr": "ASR từ token (%)"}  # fmt: skip
SCALE = {"mel": 1, "sim": 1, "digit": 100, "spk": 100, "asr": 100}


def band(ax, x, a, sty, label):
    color, marker, ls = sty
    ax.plot(x, a.mean(0), marker=marker, ls=ls, color=color, label=label, ms=4)
    ax.fill_between(x, a.min(0), a.max(0), color=color, alpha=0.15, lw=0)


def compare(names, labels, keys=("mel", "spk"), title=""):
    """Số đo theo mức bit cho vài cấu hình."""
    fig, axes = plt.subplots(1, len(keys), figsize=(5.2 * len(keys), 3.4))
    for ax, key in zip(np.atleast_1d(axes), keys, strict=True):
        for sty, name, lab in zip(STY, names, labels, strict=False):
            band(ax, BPS, SCALE[key] * seeds(name)[key], sty, lab)
        ax.set(xscale="log", xlabel="bit mỗi giây", ylabel=YL[key])
    np.atleast_1d(axes)[0].legend(fontsize=8)
    fig.suptitle(title)
    vn_axes(fig)
    plt.tight_layout()
    plt.show()


def table(names, labels, ks=(1, 2, 4, 8, 32), keys=("mel", "digit", "spk", "sim")):
    """Trung bình 3 seed (thấp nhất–cao nhất) ở vài mức bit."""
    w = max(len(lab) for lab in labels) + 2
    for key in keys:
        print(f"{YL[key]}:")
        print(" " * (w + 3) + "".join(f"{k:>18d} lớp" for k in ks))
        for name, lab in zip(names, labels, strict=True):
            a = SCALE[key] * seeds(name)[key]
            cells = []
            for k in ks:
                i = R["ks"].index(k)
                d = 1 if key in ("digit", "spk", "asr") else (3 if key == "sim" else 2)
                cells.append(f"{vn(a[:, i].mean(), d)} ({vn(a[:, i].min(), d)}–{vn(a[:, i].max(), d)})")
            print(f"   {lab:{w}s}" + "".join(f"{c:>22s}" for c in cells))


def curves(names, labels, key, k, title=""):
    """Số đo trên tập kiểm định trong lúc huấn luyện, ở k lớp."""
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for sty, name, lab in zip(STY, names, labels, strict=False):
        runs = R["codec"][name]
        steps = [c["step"] for c in runs[0]["curve"]]
        ki = (2, 8, 32).index(k)
        a = SCALE[key] * np.array([[c[key][ki] for c in r["curve"]] for r in runs])
        band(ax, steps, a, sty, lab)
    ax.set(xlabel="bước huấn luyện", ylabel=YL[key] + f", {k} lớp", title=title)
    ax.legend(fontsize=8)
    vn_axes(fig)
    plt.tight_layout()
    plt.show()

### A · Quantizer dropout: cho bao nhiêu ví dụ?

Bài báo dùng quantizer dropout cho mọi ví dụ (tỉ lệ 1) mà không so sánh với tỉ lệ khác. Ta thử 0 (luôn đủ 32 lớp),
½ (như bản phát hành của DAC) và 1, đúng thí nghiệm ở mục 6 nhưng trên bộ token hóa thật.

> **🤔 Dự đoán trước:** theo bài toán nhỏ ở mục 6, ở 1 lớp và ở 32 lớp, cấu hình nào thắng?

In [ ]:
QD = (["qdrop0", "qdrop05", "ref"], ["dropout 0", "dropout ½", "dropout 1 (bài báo)"])
compare(*QD, keys=("mel", "spk"), title="A · tỉ lệ quantizer dropout")
table(*QD)

**A: khớp, kèm một sự đánh đổi bài báo không nhắc tới.** Không có quantizer dropout, bộ token hóa gần như vô dụng khi
bị cắt: ở 1 lớp, khoảng cách mel 4,52 so với 1,98, và người chấm chỉ nghe đúng chữ số 28,8% số lần so với 81,9%.
Nhưng ở đủ 32 lớp, chính nó lại tốt nhất: mel 1,51 so với 1,59 (dải 3 seed không chồng nhau), chữ số 93,8% so với 90,3%. Đây
đúng là bài toán nhỏ ở mục 6: mô hình chỉ học với đủ 32 lớp chia việc cho cả 32 lớp, nên lớp đầu không được dạy để
tự đứng. Tỉ lệ ½ của DAC nằm giữa: gần bằng tỉ lệ 1 về mel và chữ số ở mọi mức bit, nhưng kém rõ về giọng ở 1 lớp
(58,2% so với 72,0%), còn ở 32 lớp thì khác tỉ lệ 1 chỉ trong dao động giữa các seed. Với một bộ token hóa mà mô hình
ngôn ngữ sẽ đọc với số lớp tùy ý, tỉ lệ 1 như bài báo là lựa chọn hợp lý.

### B · Huấn luyện đầu-cuối hay "partial"

Nhiều bộ token hóa trước đây huấn luyện theo giai đoạn: học xong bộ mã hóa và bộ lượng tử hóa thì đóng băng chúng,
rồi chỉ huấn luyện tiếp bộ giải mã. Bài báo (Hình 4) so hai cách và thấy cách "partial" chững lại sớm. Ta làm cùng
thí nghiệm: từ đúng điểm giữa (bước 5.000) của bộ token hóa chuẩn, một nhánh huấn luyện tiếp mọi thứ, nhánh kia đóng
băng bộ mã hóa, RVQ và mô hình ngữ nghĩa. Hai nhánh thấy đúng cùng các lô dữ liệu.

In [ ]:
PARTIAL = (["ref", "partial"], ["đầu-cuối (bài báo)", "partial (đóng băng)"])
curves(*PARTIAL, "mel", 8, "B · khoảng cách mel trên tập kiểm định")
curves(*PARTIAL, "spk", 8, "B · người chấm đúng giọng trên tập kiểm định")
table(*PARTIAL)

**B: khớp.** Hai nhánh xuất phát từ cùng một điểm ở bước 5.000. Nhánh partial vẫn tốt lên, nhưng chậm hơn hẳn, và
khoảng cách nới rộng sau mỗi lần đo: ở bước 10.000, trên tập kiểm tra, mel ở 8 lớp là 1,78 so với 1,59, người chấm
đúng giọng 83,0% so với 91,7%; dải 3 seed không chồng nhau ở mọi mức bit. Khi đóng băng bộ mã hóa và bộ lượng tử hóa,
token cũng bị đóng băng ở chất lượng của bước 5.000, và bộ giải mã chỉ còn cách dựng lại tốt nhất từ những token ấy.
Huấn luyện đầu-cuối thì vẫn sửa được cả token. Ta chỉ chạy thêm 5.000 bước, nên không biết nhánh partial có chững hẳn
như Hình 4 của bài báo hay không.

### C · Mô hình lớn hơn và mức bit

Bài báo (Hình 5) đổi độ rộng của mọi khối (từ 319 triệu đến 1,17 tỉ tham số) và thấy: mô hình lớn hơn tốt hơn, nhất là
ở mức bit cao, còn ở mức bit thấp thì số bit mới là nút thắt. Ta đổi độ rộng 32 / 64 / 96.

In [ ]:
SIZE = (["d32", "ref", "d96"], ["rộng 32", "rộng 64", "rộng 96"])
for n, lab in zip(*SIZE, strict=True):
    print(f"{lab}: {vn(R['codec'][n][0]['params'], 0)} tham số")
compare(*SIZE, keys=("mel", "spk"), title="C · độ rộng của mô hình")
table(*SIZE)

**C: khớp, nhưng bước từ rộng 64 lên 96 nằm trong dao động giữa các seed.** Ở 32 lớp, khoảng cách mel là 1,80 / 1,59 /
1,54 cho độ rộng 32 / 64 / 96; ở 1 lớp là 2,12 / 1,98 / 1,96. Mô hình lớn hơn được lợi nhiều hơn ở mức bit cao: từ
rộng 32 lên 64, mel giảm 0,22 ở 32 lớp nhưng chỉ 0,13 ở 1 lớp; từ 64 lên 96, mel giảm 0,05 ở 32 lớp nhưng chỉ 0,02 ở
1 lớp. Ở mức bit thấp, ba đường gần nhau: số bit mới là nút thắt, đúng như Hình 5. Hai lưu ý: dải 3 seed của rộng 64 và
rộng 96 chồng nhau, và mọi kích thước dùng chung η đã chọn cho rộng 64 (bài báo cũng giữ cài đặt tối ưu hóa khi đổi
kích thước). Rộng 32 thì kém hẳn về giọng (71,8% ở 32 lớp so với 92,2%).

### D · Batch lớn hơn, cùng số bước

Bài báo (Hình 6) giữ nguyên số bước và tăng batch, và thấy batch lớn hơn tốt hơn ở *mọi* thời điểm. Ta thử batch 8,
16 và 32 đoạn 1,28 giây.

In [ ]:
BATCH = (["bs8", "bs16", "ref"], ["batch 8", "batch 16", "batch 32"])
curves(*BATCH, "mel", 8, "D · khoảng cách mel trên tập kiểm định")
table(*BATCH)

**D: khớp, từ bước 2.000 trở đi.** Ở lần đo đầu tiên (bước 1.000), batch 16 còn nhỉnh hơn batch 32 trên tập kiểm định;
từ bước 2.000, batch 32 tốt hơn batch 16, và batch 16 tốt hơn batch 8, ở mọi lần đo. Ở bước 10.000, trên tập kiểm tra
ở 32 lớp: mel 2,03 / 1,80 / 1,59 và người chấm đúng giọng 44,0% / 76,7% / 92,2% cho batch 8 / 16 / 32. Giống thí
nghiệm của bài báo, phép so sánh này giữ số bước, nên batch lớn hơn cũng thấy nhiều dữ liệu hơn và tốn nhiều phép tính
hơn: batch 32 xem 320.000 đoạn, batch 8 chỉ 80.000. Nó cho thấy nên dùng batch lớn khi số bước có hạn, chứ chưa tách
được tác dụng của riêng kích thước batch.

### E · Bỏ mất mát ngữ nghĩa

Bài báo gắn một LLM 0,5 tỉ tham số để token "mang nghĩa", nhưng không thử bỏ nó đi. Ta bỏ mô hình ngữ nghĩa
(λ_sem = 0), giữ nguyên mọi thứ khác.

> **🤔 Dự đoán trước:** bỏ mất mát ngữ nghĩa, khoảng cách mel sẽ tốt lên hay tệ đi? Còn người chấm chữ số?

In [ ]:
SEM = (["sem0", "ref"], ["không có mất mát ngữ nghĩa", "λ_sem = 20 (bài báo)"])
compare(*SEM, keys=("mel", "digit"), title="E · mất mát ngữ nghĩa")
table(*SEM)

**E: tùy điều kiện.** Bỏ mất mát ngữ nghĩa, khoảng cách mel tốt hơn ở *mọi* mức bit (1,48 so với 1,59 ở 32 lớp; 1,88
so với 1,98 ở 1 lớp; dải 3 seed không chồng nhau ở mức bit nào). Từ 4 lớp trở lên, người chấm chữ số cũng nhỉnh hơn
(92,9% so với 90,3% ở 32 lớp), nhưng chỉ trong dao động giữa các seed. Ở 1 lớp thì ngược lại: người chấm chỉ nghe
đúng 71,9% so với 81,9%, và dải 3 seed không chồng nhau. Mất mát ngữ nghĩa dồn "nói gì" vào lớp token đầu tiên, và cái
giá là một phần chất lượng âm thanh. Điều đó đáng giá khi mô hình ngôn ngữ chỉ đọc vài lớp đầu, và không đáng khi ta
cần âm thanh đẹp nhất ở đủ 32 lớp. Ta chỉ thử λ_sem = 20 của bài báo; một λ nhỏ hơn có thể giữ phần lợi mà tốn ít
hơn.

### Đầu ra dạng phổ hay từng mẫu

Cuối cùng là chỗ khác bài báo ở mục 7: cùng mọi thứ, nhưng bộ giải mã cho ra trực tiếp 80 mẫu sóng âm mỗi vị trí 10 ms,
như CAT của bài báo.

In [ ]:
HEAD = (["patch", "ref"], ["cho ra từng mẫu (bài báo)", "cho ra phổ + iSTFT (ở đây)"])
compare(*HEAD, keys=("mel", "spk"), title="đầu ra của bộ giải mã")
table(*HEAD)

Ở kích thước này, đầu ra dạng phổ tốt hơn: mel 1,59 so với 1,68 ở 32 lớp (dải 3 seed không chồng nhau), người chấm
đúng giọng 92,2% so với 87,1%. Chữ số thì gần như ngang nhau (90,3% so với 90,8%). Khoảng cách nhỏ hơn ta tưởng sau
các lần chạy thử 3.000 bước: cho đủ bước, cách của bài báo cũng học được. Bài báo không cần mẹo này ở 1,6 tỉ tham số;
ở đây nó mua được chất lượng, với giá 30 ms độ trễ.

## 9 · CAT-TTS thu nhỏ và Progressive Sequence Dropout

Từ token của CAT, bài báo dựng CAT-TTS: một mô hình **hoàn toàn tự hồi quy** đọc văn bản thành giọng nói. Mỗi khung
có 32 token, nên một câu 10 giây là 125 × 32 = 4.000 token. Thay vì trải phẳng, CAT-TTS chia việc cho hai
Transformer (bố cục của RQ-Transformer, Lee và cộng sự 2022, mà Moshi gọi là Temporal + Depth):

- **Temporal Transformer** chạy theo thời gian, mỗi khung một vị trí. Đầu vào của khung t là *tổng* embedding của
  các token của nó, mỗi lớp một bảng embedding riêng (Eq. 8): ẽₜ = Σₖ Embₖ(qₜ,ₖ).
- **Depth Transformer** nhỏ hơn, chạy *bên trong* một khung: từ trạng thái của Temporal Transformer, nó đoán token
  lớp 1, rồi lớp 2 khi đã biết lớp 1, cứ thế đến lớp 32.

Mô hình TTS của bài báo cũng nhận một đoạn âm thanh mẫu: [lời của đoạn mẫu, lời cần đọc, token của đoạn mẫu] → token
của lời cần đọc, *bằng giọng của đoạn mẫu*. Bản của ta giữ nguyên cách ấy, với văn bản là hai chữ số: chữ số của đoạn
mẫu và chữ số cần đọc.

**Progressive Sequence Dropout (PSD)**, đóng góp thứ hai của bài báo: với xác suất p, một ví dụ huấn luyện chỉ giữ K̂
lớp đầu (K̂ rút đều), cả trong tổng ở đầu vào lẫn trong mất mát (Eq. 6–9); còn lại giữ đủ 32 lớp. Nhờ vậy, lúc sinh
ta chọn được số lớp K: ít lớp thì nhanh hơn và ít bit hơn, và mô hình đã từng thấy đúng tình huống ấy. Bài báo viết
K̂ ∈ {1, …, 31}; khi đó với p = 1, lớp 32 không bao giờ được học, nên ta rút K̂ ∈ {1, …, 32}.

In [ ]:
BOA, EOS_C = 10, CODES  # văn bản: chữ số 0–9 và "bắt đầu âm thanh"; lớp 1 có thêm lựa chọn "hết câu"


class TTS(nn.Module):
    """[chữ số mẫu, chữ số cần đọc, BOA] rồi các khung âm thanh. Temporal Transformer chạy theo thời gian trên tổng
    embedding của K̂ lớp đầu (Eq. 8); từ trạng thái ở khung t, Depth Transformer đoán lần lượt 32 token của khung t + 1."""

    def __init__(self, nq=NQ, d=128, layers=4, dd=64, depth_layers=2):
        super().__init__()
        self.text = nn.Embedding(11, d)
        self.emb = nn.ModuleList(nn.Embedding(CODES, d) for _ in range(nq))  # Embₖ, một bảng cho mỗi lớp
        self.temporal = nn.ModuleList(Block(d, ls=1.0) for _ in range(layers))
        self.tnorm = nn.LayerNorm(d)
        self.ctx = nn.Linear(d, dd)
        self.prev = nn.ModuleList(nn.Embedding(CODES, dd) for _ in range(nq - 1))
        self.depth = nn.ModuleList(Block(dd, ls=1.0) for _ in range(depth_layers))
        self.dnorm = nn.LayerNorm(dd)
        self.heads = nn.ModuleList(nn.Linear(dd, CODES + (k == 0)) for k in range(nq))
        self.nq, self.dh, self.ddh = nq, d // 4, dd // 4

    def frame_in(self, codes, k_hat):
        """Eq. 8: ẽₜ = Σ_{k ≤ K̂} Embₖ(q_{t,k}); mỗi ví dụ có K̂ riêng."""
        out = 0
        for k in range(int(k_hat.max())):
            out = out + self.emb[k](codes[..., k].clamp_min(0)) * (k < k_hat).float()[:, None, None]
        return out

    def temporal_states(self, text, codes, k_hat):
        h = torch.cat([self.text(text), self.frame_in(codes, k_hat)], 1)
        cs = rope(h.shape[1], self.dh)
        for b in self.temporal:
            h = b(h, cs)
        return self.tnorm(h)

    def depth_logits(self, ctx, prev, K):
        """Vị trí 1 chỉ thấy trạng thái theo thời gian; vị trí k thấy thêm token lớp k − 1 của chính khung này."""
        h = torch.stack([self.ctx(ctx)] + [self.ctx(ctx) + self.prev[k](prev[:, k]) for k in range(K - 1)], 1)
        cs = rope(K, self.ddh)
        for b in self.depth:
            h = b(h, cs)
        h = self.dnorm(h)
        return [self.heads[k](h[:, k]) for k in range(K)]

    def loss(self, text, codes, n_frames, k_hat):
        """Eq. 9: −Σₜ Σ_{k ≤ K̂} log p(q_{t,k} | văn bản, các khung trước, các lớp trước của khung t), chia cho số khung.
        codes (B, T, 32): khung mẫu rồi khung cần đọc nối nhau (−1 là phần đệm); sau khung cuối là "hết câu"."""
        h = self.temporal_states(text, codes[:, :-1], k_hat)
        ctx = h[:, 2:]  # trạng thái ở BOA đoán khung 1, ở khung t đoán khung t + 1
        t_idx = torch.arange(ctx.shape[1])[None]
        valid, is_eos = t_idx <= n_frames[:, None], t_idx == n_frames[:, None]
        target = codes[:, : ctx.shape[1]].clone()
        target[..., 0] = torch.where(is_eos, torch.tensor(EOS_C), target[..., 0])
        sel = valid.reshape(-1)
        c, tg = ctx.reshape(-1, ctx.shape[-1])[sel], target.reshape(-1, self.nq)[sel]
        kk = k_hat[:, None].expand(-1, ctx.shape[1]).reshape(-1)[sel]
        eos = is_eos.reshape(-1)[sel]
        K = int(kk.max())
        logits = self.depth_logits(c, tg[:, : K - 1].clamp(0, CODES - 1), K)
        total, count = 0.0, 0
        for k in range(K):
            m = (k < kk) & (~eos | (k == 0))  # chỉ các lớp được giữ; khung "hết câu" chỉ có lớp 1
            total = total + F.cross_entropy(logits[k][m], tg[m, k], reduction="sum")
            count += int(m.sum())
        return total / sel.sum(), count


tts = TTS()
print(
    f"CAT-TTS thu nhỏ: {vn(sum(p.numel() for p in tts.parameters()), 0)} tham số (chưa huấn luyện, chỉ để xem hình dạng)"
)

Ô dưới lấy 8 cặp bản ghi huấn luyện (cùng người đọc), mã hóa bằng CAT ở trên, rồi rút K̂ cho từng cặp như PSD với
p = 0, ½ và 1. Hình vẽ lớp nào được giữ (xanh) trong tổng ở đầu vào và trong mất mát. Mô hình chưa học gì đoán đều
1.024 mã, nên mất mát của nó xấp xỉ ln 1.024 ≈ 6,93 nat cho mỗi token được giữ: một cách kiểm tra rằng mất mát chỉ
đếm các lớp được giữ.

In [ ]:
pairs = [(TRAIN[i], TRAIN[i + 1]) for i in range(0, 160, 20)]
assert all(spk[a] == spk[b] and a != b for a, b in pairs)
clips = sorted({c for pr in pairs for c in pr})
with torch.no_grad():
    enc = codec.encode(pad_clips(clips))
tok = {c: enc[i, : -(-n_samples[c] // HOP)] for i, c in enumerate(clips)}
seqs = [torch.cat([tok[a], tok[b]]) for a, b in pairs]
n_fr = torch.tensor([len(q) for q in seqs])
batch_codes = torch.full((len(pairs), int(n_fr.max()) + 1, NQ), -1, dtype=torch.long)
for i, q in enumerate(seqs):
    batch_codes[i, : len(q)] = q
batch_text = torch.tensor([[digit[a], digit[b], BOA] for a, b in pairs])

rng_psd = np.random.default_rng(0)
fig, axes = plt.subplots(1, 3, figsize=(12, 2.8), sharey=True)
for ax, p in zip(axes, (0.0, 0.5, 1.0), strict=True):
    z = rng_psd.random(len(pairs)) < p  # Eq. 6
    k_hat = torch.tensor(np.where(z, rng_psd.integers(1, NQ + 1, len(pairs)), NQ))  # Eq. 7
    with torch.no_grad():
        loss, count = tts.loss(batch_text, batch_codes, n_fr, k_hat)
    per_token = float(loss) * int(n_fr.sum() + len(pairs)) / count
    print(f"p = {vn(p, 1)}: K̂ = {k_hat.tolist()}; {vn(count, 0)} token trong mất mát, {vn(per_token, 2)} nat mỗi token")
    assert abs(per_token - math.log(CODES)) < 0.7
    ax.imshow(np.arange(NQ)[None] < k_hat.numpy()[:, None], cmap=plt.matplotlib.colors.ListedColormap(["#e6e9ef", BLUE]),
              aspect="auto", extent=[0.5, NQ + 0.5, len(pairs) + 0.5, 0.5])  # fmt: skip
    ax.set(title=f"p = {vn(p, 1)}", xlabel="lớp RVQ")
axes[0].set_ylabel("ví dụ huấn luyện")
vn_axes(fig)
plt.tight_layout()
plt.show()

### Kết quả: 4 giá trị p × 3 seed

Script đã huấn luyện CAT-TTS thu nhỏ trên token của bộ token hóa seed 0, với p = 0, ¼, ½ và 1, mỗi giá trị 3 seed,
4.000 bước, batch 64 cặp. Một mẹo cần thiết ở kích thước này: mỗi bản ghi được mã hóa thêm 7 lần, mỗi lần
lùi thêm 10 ms trước khi mã hóa, nên ranh giới khung rơi vào chỗ khác và token khác đi. Không có nó, mô hình học
thuộc lòng token của 2.397 bản ghi huấn luyện. Lúc kiểm tra, mỗi bản ghi kiểm tra làm đoạn mẫu, rồi mô hình đọc một
chữ số *khác* (chọn trước, cố định) ở K = 1, 2, 4, 8, 16 và 32 lớp, lấy mẫu với nhiệt độ 0,8 và top-k 50 (chọn trước,
không dò). Người chấm chữ số thay cho WER; người chấm giọng và SIM (so với đoạn mẫu) thay cho SIM của bài báo.

Hai mốc so sánh: **"bản ghi thật"** là chính người đọc của đoạn mẫu đọc đúng chữ số cần đọc (một bản ghi kiểm tra
khác), cho qua bộ token hóa ở cùng K lớp: mức tốt nhất một TTS hoàn hảo có thể đạt. Và SIM giữa hai bản ghi thật
*cùng* người đọc, so với hai bản ghi của hai người *khác nhau*.

> **🤔 Dự đoán trước:** ở 32 lớp, mô hình p = 0 và p = 1 khác nhau nhiều không? Còn ở 1 lớp?

In [ ]:
TT, BASE = R["tts"], R["baselines"]
TKS = np.array(BASE["oracle"]["k"])
PS_ = ["0", "0.25", "0.5", "1"]
PLAB = {"0": "p = 0 (không PSD)", "0.25": "p = ¼", "0.5": "p = ½", "1": "p = 1"}
assert TT["1"][0]["params"] == sum(p.numel() for p in tts.parameters()) and R["ttsSteps"] == 4000

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, key in zip(axes, ("digit", "spk", "sim"), strict=True):
    sc = 1 if key == "sim" else 100
    for sty, p in zip(STY, PS_, strict=True):
        band(ax, TKS, sc * np.array([r["test"][key] for r in TT[p]]), sty, PLAB[p])
    ax.plot(TKS, sc * np.array(BASE["oracle"][key]), color="k", lw=1, label="bản ghi thật qua bộ token hóa")
    if key == "sim":
        for kind, lab, ls in (("same", "hai bản ghi thật, cùng người", "--"), ("other", "hai người khác nhau", ":")):
            ax.axhline(BASE["real"][kind]["sim"], color=GREY, ls=ls, lw=1, label=lab)
    else:
        ax.axhline(100 / (10 if key == "digit" else len(SPEAKERS)), color=GREY, ls=":", lw=1, label="đoán bừa")
    ax.set(xscale="log", xlabel="số lớp K lúc sinh", ylabel={"digit": YL["digit"], "spk": YL["spk"], "sim": "SIM"}[key])
    ax.xaxis.set_major_locator(FixedLocator(list(TKS)))
h0, l0 = axes[0].get_legend_handles_labels()
h2, l2 = axes[2].get_legend_handles_labels()
fig.legend(h0 + h2[len(l0) - 1 :], l0 + l2[len(l0) - 1 :], loc="lower center", ncol=4, fontsize=8, frameon=False)
fig.suptitle("F · CAT-TTS thu nhỏ: đọc chữ số khác bằng giọng của đoạn mẫu")
vn_axes(fig)
for ax in axes:
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _p: f"{v:g}"))
plt.tight_layout(rect=(0, 0.16, 1, 1))
plt.show()

TNAME = {"digit": "người chấm đúng chữ số", "spk": "người chấm đúng giọng", "sim": "SIM so với đoạn mẫu",
         "no_end": "câu không kết thúc"}  # fmt: skip
for key, name in TNAME.items():
    f = (lambda v: vn(v, 3)) if key == "sim" else pct
    print(f"\n{name}, trung bình 3 seed (thấp nhất–cao nhất)")
    print(" K | " + " | ".join(f"{PLAB[p]:>21s}" for p in PS_) + (" | bản ghi thật" if key != "no_end" else ""))
    for i, k in enumerate(TKS):
        c = [np.array([r["test"][key][i] for r in TT[p]]) for p in PS_]
        row = " | ".join(f"{f(x.mean())} ({f(x.min())}–{f(x.max())})".rjust(21) for x in c)
        print(f"{k:2d} | {row}" + (f" | {f(BASE['oracle'][key][i])}" if key != "no_end" else ""))
print(f"\nSIM giữa hai bản ghi thật: cùng người {vn(BASE['real']['same']['sim'], 3)}, "
      f"hai người khác nhau {vn(BASE['real']['other']['sim'], 3)}")  # fmt: skip

**F: khớp về độ rõ chữ; về giọng, mô hình nào cũng chỉ bắt chước được một phần.**

- **Không có PSD, bớt lớp lúc sinh làm mô hình đọc sai.** Ở 1 lớp, mô hình p = 0 đọc đúng chữ số 69,8% số lần, còn các
  mô hình có PSD đọc đúng 80,1–83,1% (dải 3 seed không chồng nhau). Ở 1 lớp, trung bình 13,4% câu của p = 0 không bao
  giờ kết thúc, nhưng con số này dao động rất mạnh giữa các seed (0,3–27,3%); có PSD, nó chỉ 0,8–1,3%. Mô hình p = 0
  chỉ từng thấy ở đầu vào tổng embedding của đủ 32 lớp; tổng của 1 lớp là thứ nó chưa gặp bao giờ.
- **Có PSD, các đường gần như nằm ngang.** Đọc với 1 lớp tốt ngang 32 lớp, và gần bằng bản ghi thật qua bộ token hóa
  ở cùng 1 lớp (83,1% với p = 1, so với 84,3%). Ở 1 lớp, Depth Transformer chỉ chạy 1 bước mỗi khung thay vì 32.
- **Ở 32 lớp, khoảng cách nhỏ lại:** 74,9% với p = 0, so với 77,1–81,0% với PSD; chỉ p = 1 tách hẳn khỏi dải seed của
  p = 0. Ở mọi p, sinh đủ 32 lớp không tốt hơn sinh 8–16 lớp, và còn xa bản ghi thật (91,7%).
- **Giọng:** với PSD, người chấm nhận ra đúng người đọc của đoạn mẫu 35–41% số lần ở mọi K. Đoán bừa đạt 16,7%, còn
  bản ghi thật qua bộ token hóa đạt 66–92%. SIM 0,86–0,88 nằm giữa hai bản ghi thật của cùng một người (0,973) và của
  hai người khác nhau (0,711). Mô hình p = 0 kém hơn ở 1 lớp (21,4%), còn ở 32 lớp thì ngang các mô hình có PSD
  (SIM 0,873 so với 0,861–0,869, trong dao động giữa các seed). Bài báo thấy PSD giữ được cả WER lẫn SIM. Ở đây PSD
  giữ được độ rõ chữ, còn việc bắt chước giọng thì yếu ở mọi p, dù cả 6 người đọc đều đã có trong dữ liệu huấn luyện.

Thứ hạng giữa p = ¼, ½ và 1 phần lớn nằm trong dao động giữa các seed: ở kích thước này, chỉ cần *một phần* ví dụ bị
cắt là đủ.

### Nghe CAT-TTS

Token của vài câu mô hình đã sinh (seed 0) có sẵn trong tệp dữ liệu; ô dưới giải mã chúng bằng bộ token hóa ở mục 7,
chấm lại bằng hai người chấm và so với kết quả của script. Mỗi đoạn mẫu là người đọc thứ s đọc chữ số s; mô hình phải
đọc tiếp một chữ số khác bằng giọng ấy.

In [ ]:
SH = W["show"]
shown = {}
for p in ("0", "1"):
    s = SH[p]
    prompt_x = pad_clips(s["ids"])
    for ki, k in enumerate(s["k"]):
        n = [len(c) for c in s["codes"][ki]]
        padded = torch.zeros(len(n), WIN, k, dtype=torch.long)
        for j, c in enumerate(s["codes"][ki]):
            if n[j]:
                padded[j, : n[j]] = torch.tensor(c)
        with torch.no_grad():
            y = codec.decode(padded) * (torch.arange(WIN * HOP)[None] < torch.tensor(n)[:, None] * HOP).float()
            ok_d = (jd(y).argmax(-1).numpy() == np.array(s["target"])).astype(int)
            ok_s = (js(y).argmax(-1).numpy() == spk[s["ids"]]).astype(int)
            sim_ = F.cosine_similarity(js.embed(y), js.embed(prompt_x)).numpy()
        assert (ok_d != s["digit"][ki]).sum() <= 1 and (ok_s != s["spk"][ki]).sum() <= 1
        assert np.allclose(sim_, s["sim"][ki], atol=2e-3)
        shown[p, k] = (y, n)
        print(
            f"{PLAB[p]:18s} {k:2d} lớp: đúng chữ số {ok_d.sum()}/6, đúng giọng {ok_s.sum()}/6, SIM {vn(sim_.mean(), 3)}"
        )

for j in (0, 4):
    c = SH["1"]["ids"][j]
    play(waves[c], f'đoạn mẫu: "{SPEAKERS[spk[c]]}" đọc {digit[c]}; mô hình cần đọc {SH["1"]["target"][j]}')
    for p, k in (("0", 32), ("1", 32), ("0", 2), ("1", 2)):
        y, n = shown[p, k]
        play(y[j, : n[j] * HOP].numpy(), f"   {PLAB[p]}, {k} lớp")

## 10 · Thử thay đổi

### Lớp nào mang gì

RVQ cộng phần đóng góp của từng lớp. Ô dưới giải mã chữ "seven" từ chỉ lớp 1, từ lớp 2–32 mà *bỏ* lớp 1, và từ đủ
32 lớp. Hãy nghe, và thử bỏ lớp khác.

In [ ]:
with torch.no_grad():
    parts = [lay.outp(lay.book[codes_full[..., i]]) for i, lay in enumerate(codec.rvq.layers)]
    for name, q in (("chỉ lớp 1", parts[0]), ("lớp 2–32, bỏ lớp 1", sum(parts[1:])), ("đủ 32 lớp", sum(parts))):
        y = codec.decode_q(q)[:, : len(x_ex)]
        print(f"{name:20s} mất mát mel {vn(mel_loss(y, x0), 2)}")
        play(y[0], name)

### Một bản ghi khác

Đổi người đọc, chữ số và số lớp rồi chạy lại.

In [ ]:
NGUOI, CHU_SO, SO_LOP = "jackson", 3, 2
c = TEST[(spk[TEST] == SPEAKERS.index(NGUOI)) & (digit[TEST] == CHU_SO)][0]
with torch.no_grad():
    y = codec.decode(codec.encode(pad_clips([c]), SO_LOP))[0, : len(waves[c])]
    guess_d = int(jd(y[None]).argmax())
    guess_s = SPEAKERS[int(js(y[None]).argmax())]
play(waves[c], f'bản gốc: "{NGUOI}" đọc {CHU_SO}')
play(
    y, f"{SO_LOP} lớp ({vn(FRAME_RATE * SO_LOP * BITS, 0)} bit/s): người chấm nghe ra chữ số {guess_d}, giọng {guess_s}"
)

### Mất mát mel với ít cửa sổ hơn

Ô dưới tính lại mất mát mel của năm biến thể ở mục 5, lần lượt chỉ với cửa sổ ngắn nhất và chỉ với cửa sổ dài nhất.
Biến thể nào đổi thứ hạng?

In [ ]:
for wins in ((32, 64), (1024, 2048)):
    sub = [(w, b) for w, b in zip(MEL_WINDOWS, MEL_BANDS, strict=True) if w in wins]
    vals = {}
    for name, y in variants.items():
        vals[name] = sum(
            float((mel(y, w, b).clamp_min(1e-5).log10() - mel(x0, w, b).clamp_min(1e-5).log10()).abs().mean())
            for w, b in sub
        )
    print(f"cửa sổ {wins}: " + " < ".join(f"{n} {vn(v, 2)}" for n, v in sorted(vals.items(), key=lambda kv: kv[1])))

## 11 · Giới hạn

- **Nhỏ hơn bài báo hàng nghìn lần.** 1,2 triệu tham số thay vì 1,6 tỉ; 21,9 phút chữ số tiếng Anh của 6 người thay vì
  3 triệu giờ tiếng nói, âm thanh và nhạc; 8 kHz thay vì 24 kHz. Chất lượng tuyệt đối không so được, và ngay ở 32 lớp
  các họa âm cao vẫn bị nhòe.
- **Không có GAN, một giai đoạn huấn luyện.** Bài báo huấn luyện 520.000 bước không có GAN rồi 500.000 bước có GAN
  (MPD và bộ phân biệt phổ phức). Ở ngân sách của ta, 1.500 bước GAN kém hơn cùng lượng tính toán dùng cho mất mát mel,
  nên ta bỏ GAN. Mất mát mel không nhìn pha, nên chính phần GAN mới dạy bộ giải mã dựng pha đúng.
- **Chưa hội tụ.** Khoảng cách mel trên tập kiểm định vẫn giảm ở bước 10.000. Mọi kết luận là về cùng một ngân sách
  10.000 bước, và η = 3 · 10⁻³ được chọn cho cấu hình chuẩn rồi dùng chung cho mọi cấu hình.
- **Người chấm nhỏ.** Hai CNN học trên 2.397 bản ghi thay cho mô hình nhận dạng tiếng nói và mô hình xác minh người nói
  của bài báo. "Đúng giọng" ở đây là chọn đúng một trong 6 người đã biết, dễ hơn nhiều so với nhận ra một người bất kỳ.
- **TTS không phải zero-shot.** Bài báo sao chép giọng của người chưa từng gặp. Ở đây cả 6 người đều có trong tập huấn
  luyện, và văn bản chỉ là một chữ số. Kết quả về giọng vì vậy là một cận trên rộng rãi, mà vẫn yếu.
- **Hai chỗ khác bài báo:** đầu ra dạng phổ (30 ms độ trễ; mục 8 so với cách của bài báo) và K̂ rút trong {1, …, 32}
  thay vì {1, …, 31}.
- **3 seed.** Những khác biệt có dải seed chồng nhau được ghi là trong dao động giữa các seed.

## Tóm tắt

- **Một bộ token hóa âm thanh** gồm bộ mã hóa, bộ lượng tử hóa và bộ giải mã. CAT cắt sóng âm thành 12,5 khung mỗi
  giây, mỗi khung 32 token, mỗi token 10 bit: tối đa 4.000 bit/s, nhỏ hơn tệp wav 24 kHz 96 lần.
- **Thước đo phải giống tai.** Đảo dấu cả sóng âm cho MSE lớn nhất mà tai không nghe khác; mất mát mel đa tỉ lệ so
  độ lớn của phổ ở nhiều độ dài cửa sổ.
- **RVQ** lượng tử hóa phần dư lớp sau lớp; gradient đi qua argmax nhờ straight-through. **Quantizer dropout** dạy
  các lớp đầu tự đứng, nên một bộ token hóa dùng được ở mọi mức bit.
- **CAT thu nhỏ** (1.166.882 tham số): 1 lớp (125 bit/s) đủ để nghe ra chữ số 81,9% số lần, giọng cần khoảng 4 lớp, và
  từ 8 lớp trở đi không khá hơn nữa. Các lớp cuối dùng ít mã: khoảng 3.300 bit/s là thông tin thật.
- **Các nhận định:** A khớp, kèm đánh đổi: không có dropout thì tốt nhất ở 32 lớp, vô dụng ở 1 lớp. B khớp: huấn luyện
  đầu-cuối bỏ xa partial. C khớp: mô hình lớn được lợi nhiều hơn ở mức bit cao, dù bước 64 → 96 nằm trong dao động
  giữa các seed. D khớp: batch lớn hơn tốt hơn ở cùng số bước. E tùy điều kiện: mất mát ngữ nghĩa giúp ở 1 lớp, tốn
  chất lượng ở mọi mức bit. F khớp về độ rõ chữ: PSD giữ TTS tốt khi bớt lớp, nhưng giọng chỉ bắt chước được một
  phần ở mọi p.

## Câu hỏi ôn tập

1. Đảo dấu cả sóng âm cho MSE / công suất bằng 4, mà mất mát mel bằng 0. Mất mát mel bỏ qua điều gì? Vì sao đó vừa là
   ưu điểm vừa là chỗ thiếu?
2. Bộ token hóa không có quantizer dropout cắt về 1 lớp cho mel 4,52, dù lớp 1 được huấn luyện ở mọi ví dụ. Vì sao?
3. Một bộ token hóa 12,5 khung mỗi giây, bảng mã 4.096 mã, giữ 8 lớp. Bao nhiêu bit mỗi giây? Nếu trải phẳng cho một
   mô hình ngôn ngữ, câu nói 10 giây dài bao nhiêu token?
4. Vì sao CAT-TTS dùng một Temporal Transformer và một Depth Transformer thay vì một Transformer đọc 32 lớp trải phẳng?
5. Mô hình TTS p = 0 sinh ở 1 lớp không kết thúc được trung bình 13,4% số câu. Vì sao, khi lựa chọn "hết câu" nằm ngay ở lớp 1?
6. Thí nghiệm D giữ số bước và tăng batch. Hãy thiết kế một thí nghiệm tách được tác dụng của kích thước batch khỏi
   lượng dữ liệu đã xem.

<details><summary>Gợi ý đáp án</summary>

1. Mất mát mel chỉ so *độ lớn* của phổ trong từng cửa sổ, không so pha. Đảo dấu chỉ cộng π vào pha của mọi tần số,
   nên mọi độ lớn giữ nguyên, và tai cũng không nghe ra. Đó là ưu điểm: mô hình không bị phạt vì những khác biệt vô
   hại. Chỗ thiếu: mất mát mel không dạy bộ giải mã chọn pha sao cho các cửa sổ cộng chồng khớp nhau. Bài báo thêm các
   bộ phân biệt GAN nhìn thẳng vào sóng âm và phổ phức để lo phần đó.
2. Khi luôn có đủ 32 lớp, bộ giải mã chỉ từng thấy tổng của cả 32 lớp, và không có gì buộc lớp 1 phải tự mang phần
   quan trọng nhất, vì các lớp sau luôn có mặt để sửa. Cắt về 1 lớp đưa vào bộ giải mã một vectơ nó chưa gặp bao giờ.
3. log₂ 4.096 = 12 bit, nên 12,5 · 8 · 12 = 1.200 bit/s. Trải phẳng là 12,5 · 8 = 100 token mỗi giây, tức 1.000 token
   cho 10 giây.
4. Trải phẳng 32 lớp, 10 giây đã là 4.000 token, và chi phí chú ý tăng theo bình phương độ dài. Temporal Transformer
   chỉ thấy 125 vị trí cho 10 giây; Depth Transformer nhỏ, chạy trên 32 vị trí bên trong một khung, và lo quan hệ giữa
   các lớp. Nhờ cách chia này, PSD cũng chỉ cần bỏ bớt các vị trí cuối của Depth Transformer và bớt số hạng trong tổng
   embedding.
5. Mô hình đoán "hết câu" từ trạng thái của Temporal Transformer, mà trạng thái ấy được tính từ tổng embedding của các
   khung trước. Mô hình p = 0 chỉ học với tổng của đủ 32 lớp; ở 1 lớp, tổng ấy khác hẳn, nên các dấu hiệu "câu đã đọc
   xong" không còn ở chỗ quen thuộc. Với PSD, mô hình đã thấy đúng loại đầu vào này lúc học.
6. Ví dụ: batch 8 chạy 40.000 bước so với batch 32 chạy 10.000 bước, để cả hai xem cùng 320.000 đoạn, và chọn η riêng
   cho từng batch trên tập kiểm định. So ở cùng lượng dữ liệu (và ghi cả thời gian chạy) sẽ cho biết batch lớn có
   thật sự tốt hơn cho mỗi ví dụ, hay chỉ tốt hơn cho mỗi bước.
</details>

## Tìm hiểu thêm

- Gong và cộng sự (2026), *MOSS-Audio-Tokenizer: Scaling Audio Tokenizers for Future Audio Foundation Models*:
  [arXiv:2602.10934](https://arxiv.org/abs/2602.10934); mã và trọng số:
  [OpenMOSS/MOSS-Audio-Tokenizer](https://github.com/OpenMOSS/MOSS-Audio-Tokenizer).
- van den Oord, Vinyals & Kavukcuoglu (2017), *Neural Discrete Representation Learning* (VQ-VAE):
  [arXiv:1711.00937](https://arxiv.org/abs/1711.00937).
- Zeghidour và cộng sự (2021), *SoundStream: An End-to-End Neural Audio Codec*:
  [arXiv:2107.03312](https://arxiv.org/abs/2107.03312).
- Défossez và cộng sự (2022), *High Fidelity Neural Audio Compression* (EnCodec):
  [arXiv:2210.13438](https://arxiv.org/abs/2210.13438).
- Kumar và cộng sự (2023), *High-Fidelity Audio Compression with Improved RVQGAN* (DAC):
  [arXiv:2306.06546](https://arxiv.org/abs/2306.06546).
- Kong, Kim & Bae (2020), *HiFi-GAN*: [arXiv:2010.05646](https://arxiv.org/abs/2010.05646).
- Siuzdak (2023), *Vocos: Closing the gap between time-domain and Fourier-based neural vocoders*:
  [arXiv:2306.00814](https://arxiv.org/abs/2306.00814).
- Lee và cộng sự (2022), *Autoregressive Image Generation using Residual Quantization* (RQ-Transformer):
  [arXiv:2203.01941](https://arxiv.org/abs/2203.01941).
- Défossez và cộng sự (2024), *Moshi: a speech-text foundation model for real-time dialogue*:
  [arXiv:2410.00037](https://arxiv.org/abs/2410.00037).
- Kyutai (2025), *Neural audio codecs: how to get audio into LLMs*:
  [kyutai.org/codec-explainer](https://kyutai.org/codec-explainer/).
- Free Spoken Digit Dataset: [github.com/Jakobovski/free-spoken-digit-dataset](https://github.com/Jakobovski/free-spoken-digit-dataset).

**Quay lại:** [Bài 14 · Token & embedding](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-embedding/) ·
[Bài 19 · Autoencoder & VAE](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/autoencoder-vae/).

In [ ]:
import platform

import matplotlib

print(
    f"Python {platform.python_version()} · numpy {np.__version__} · torch {torch.__version__} · "
    f"matplotlib {matplotlib.__version__}"
)